In [2]:
import requests
import json
from datetime import datetime, timezone
import os
import uuid

print("Libraries imported successfully!")

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 4, Finished, Available, Finished, False)

Libraries imported successfully!


In [3]:
base_url = "https://hapi.fhir.org/baseR4"

api_url = f"{base_url}/Patient"

params = {
    "_count": 10,
    "_format": "json"
}

response = requests.get(
    api_url,
    params=params,
    timeout=60
)

response.raise_for_status()

patient_data = response.json()

print("API Status:", response.status_code)
print(
    "Patients fetched:",
    len(patient_data.get("entry", []))
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 5, Finished, Available, Finished, False)

API Status: 200
Patients fetched: 10


In [4]:
load_date = datetime.now(
    timezone.utc
).strftime("%Y-%m-%d")

run_id = str(uuid.uuid4())

folder_path = (
    "/lakehouse/default/Files/raw/Patient/"
    f"load_date={load_date}/run_id={run_id}"
)

os.makedirs(folder_path, exist_ok=True)

file_path = f"{folder_path}/page_1.json"

with open(file_path, "w", encoding="utf-8") as f:
    json.dump(patient_data, f, indent=2)

print("JSON saved successfully!")
print(file_path)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 6, Finished, Available, Finished, False)

JSON saved successfully!
/lakehouse/default/Files/raw/Patient/load_date=2026-10-25/run_id=d227640b-3866-48a5-9e2b-8cb19a448438/page_1.json


In [5]:
import requests
import json
import os
import uuid
import time
from datetime import datetime, timezone

BASE_URL = "https://hapi.fhir.org/baseR4"

RESOURCES = [
    "Patient",
    "Encounter",
    "Observation",
    "Condition"
]

PAGE_SIZE = 100

def ingest_fhir_resource(resource):

    url = f"{BASE_URL}/{resource}"
    params = {
        "_count": PAGE_SIZE,
        "_format": "json"
    }

    load_date = datetime.now(
        timezone.utc
    ).strftime("%Y-%m-%d")

    run_id = str(uuid.uuid4())

    folder_path = (
        f"/lakehouse/default/Files/raw/{resource}/"
        f"load_date={load_date}/run_id={run_id}"
    )

    os.makedirs(folder_path, exist_ok=True)

    page_number = 0
    total_records = 0

    while url:

        response = requests.get(
            url,
            params=params,
            timeout=60
        )
        response.raise_for_status()

        bundle = response.json()
        page_number += 1

        file_path = (
            f"{folder_path}/page_{page_number}.json"
        )

        with open(
            file_path, "w", encoding="utf-8"
        ) as f:
            json.dump(bundle, f, indent=2)

        records = bundle.get("entry", [])
        total_records += len(records)

        print(
            f"{resource} | Page {page_number} | "
            f"Records: {len(records)}"
        )

        next_url = next(
            (
                link["url"]
                for link in bundle.get("link", [])
                if link.get("relation") == "next"
            ),
            None
        )

        url = next_url
        params = None

        if url:
            time.sleep(0.5)

    print(
        f"Completed {resource}: "
        f"{page_number} pages, "
        f"{total_records} records"
    )

    return {
        "resource": resource,
        "pages": page_number,
        "records": total_records,
        "run_id": run_id,
        "folder": folder_path
    }


results = []

for resource in RESOURCES:
    try:
        result = ingest_fhir_resource(resource)
        results.append(result)
    except Exception as e:
        print(f"FAILED: {resource} | {str(e)}")
        raise

print("All four resources processed successfully.")

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 7, Finished, Available, Finished, False)

Patient | Page 1 | Records: 100
Patient | Page 2 | Records: 100
Patient | Page 3 | Records: 100
Patient | Page 4 | Records: 100
Patient | Page 5 | Records: 100
Patient | Page 6 | Records: 100
Patient | Page 7 | Records: 100
Patient | Page 8 | Records: 100
Patient | Page 9 | Records: 100
Patient | Page 10 | Records: 100
Patient | Page 11 | Records: 100
Patient | Page 12 | Records: 100
Patient | Page 13 | Records: 100
Patient | Page 14 | Records: 100
Patient | Page 15 | Records: 100
Patient | Page 16 | Records: 100
Patient | Page 17 | Records: 100
Patient | Page 18 | Records: 100
Patient | Page 19 | Records: 100
Patient | Page 20 | Records: 100
Patient | Page 21 | Records: 100
Patient | Page 22 | Records: 100
Patient | Page 23 | Records: 100
Patient | Page 24 | Records: 100
Patient | Page 25 | Records: 100
Patient | Page 26 | Records: 100
Patient | Page 27 | Records: 100
Patient | Page 28 | Records: 100
Patient | Page 29 | Records: 100
Patient | Page 30 | Records: 100
Patient | Page 31 |

In [7]:

from notebookutils import mssparkutils

raw_path = "Files/raw/Patient"

# List files and folders inside Patient
items = mssparkutils.fs.ls(raw_path)

for item in items:
    print(item.path, "| Directory:", item.isDir)


StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 9, Finished, Available, Finished, False)

abfss://d621f477-fcf6-44f4-b9da-2e279f03b748@onelake.dfs.fabric.microsoft.com/7c5272cd-1a9e-4e1a-88c5-395a8938a85a/Files/raw/Patient/load_date=2026-10-25 | Directory: True


In [8]:

from notebookutils import mssparkutils

def list_files(path):
    for item in mssparkutils.fs.ls(path):
        if item.isDir:
            list_files(item.path)
        elif item.path.endswith(".json"):
            print(item.path)

list_files("Files/raw/Patient")


StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 10, Finished, Available, Finished, False)

abfss://d621f477-fcf6-44f4-b9da-2e279f03b748@onelake.dfs.fabric.microsoft.com/7c5272cd-1a9e-4e1a-88c5-395a8938a85a/Files/raw/Patient/load_date=2026-10-25/run_id=2243bed4-2e0f-470a-be84-00d883f472ab/page_1.json
abfss://d621f477-fcf6-44f4-b9da-2e279f03b748@onelake.dfs.fabric.microsoft.com/7c5272cd-1a9e-4e1a-88c5-395a8938a85a/Files/raw/Patient/load_date=2026-10-25/run_id=2243bed4-2e0f-470a-be84-00d883f472ab/page_10.json
abfss://d621f477-fcf6-44f4-b9da-2e279f03b748@onelake.dfs.fabric.microsoft.com/7c5272cd-1a9e-4e1a-88c5-395a8938a85a/Files/raw/Patient/load_date=2026-10-25/run_id=2243bed4-2e0f-470a-be84-00d883f472ab/page_11.json
abfss://d621f477-fcf6-44f4-b9da-2e279f03b748@onelake.dfs.fabric.microsoft.com/7c5272cd-1a9e-4e1a-88c5-395a8938a85a/Files/raw/Patient/load_date=2026-10-25/run_id=2243bed4-2e0f-470a-be84-00d883f472ab/page_12.json
abfss://d621f477-fcf6-44f4-b9da-2e279f03b748@onelake.dfs.fabric.microsoft.com/7c5272cd-1a9e-4e1a-88c5-395a8938a85a/Files/raw/Patient/load_date=2026-10-25/run

In [9]:

from notebookutils import mssparkutils
from pyspark.sql import functions as F

def get_json_files(path):
    files = []
    for item in mssparkutils.fs.ls(path):
        if item.isDir:
            files.extend(get_json_files(item.path))
        elif item.path.endswith(".json"):
            files.append(item.path)
    return files

patient_files = get_json_files("Files/raw/Patient")

print("Total JSON files:", len(patient_files))
print("First 3 files:", patient_files[:3])

# Read JSON files using their exact paths
patient_raw_df = (
    spark.read
    .option("multiLine", "true")
    .json(patient_files)
)

patient_raw_df.printSchema()
patient_raw_df.show(3, truncate=False)


StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 11, Finished, Available, Finished, False)

Total JSON files: 92
First 3 files: ['abfss://d621f477-fcf6-44f4-b9da-2e279f03b748@onelake.dfs.fabric.microsoft.com/7c5272cd-1a9e-4e1a-88c5-395a8938a85a/Files/raw/Patient/load_date=2026-10-25/run_id=2243bed4-2e0f-470a-be84-00d883f472ab/page_1.json', 'abfss://d621f477-fcf6-44f4-b9da-2e279f03b748@onelake.dfs.fabric.microsoft.com/7c5272cd-1a9e-4e1a-88c5-395a8938a85a/Files/raw/Patient/load_date=2026-10-25/run_id=2243bed4-2e0f-470a-be84-00d883f472ab/page_10.json', 'abfss://d621f477-fcf6-44f4-b9da-2e279f03b748@onelake.dfs.fabric.microsoft.com/7c5272cd-1a9e-4e1a-88c5-395a8938a85a/Files/raw/Patient/load_date=2026-10-25/run_id=2243bed4-2e0f-470a-be84-00d883f472ab/page_11.json']
root
 |-- entry: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- fullUrl: string (nullable = true)
 |    |    |-- resource: struct (nullable = true)
 |    |    |    |-- _birthDate: struct (nullable = true)
 |    |    |    |    |-- extension: array (nullable = true)
 |    |    |    |

In [10]:

from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# Extract individual FHIR resources from each Bundle
patient_bronze_df = (
    patient_raw_df
    .select(
        F.explode_outer("entry").alias("entry"),
        F.current_timestamp().alias("extraction_timestamp")
    )
    .select(
        F.col("entry.fullUrl").alias("full_url"),
        F.col("entry.resource").alias("resource"),
        F.col("extraction_timestamp")
    )
    .filter(F.col("resource").isNotNull())
    .withColumn("resource_id", F.col("resource.id"))
    .withColumn("resource_type", F.col("resource.resourceType"))
    .withColumn("api_url_or_params", F.lit(
        "https://hapi.fhir.org/baseR4/Patient"
    ))
)

print("Extracted Patient records:", patient_bronze_df.count())
patient_bronze_df.show(5, truncate=False)


StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 12, Finished, Available, Finished, False)

Extracted Patient records: 9041
+------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------+-----------+-------------+------------------------------------+
|full_url                                  |resource                                                                                                                                                                                                                                                                                                                                                                                     |extraction_timestamp     |

In [11]:
table_name = "Bronze_Patient"

if not spark.catalog.tableExists(table_name):
    (
        patient_bronze_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(table_name)
    )
    print(f"{table_name} table created successfully!")
else:
    print(f"{table_name} already exists. Skipping write to avoid duplicates.")

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 13, Finished, Available, Finished, False)

Bronze_Patient table created successfully!


In [12]:
print("Total Bronze Patient records:")
print(spark.table("Bronze_Patient").count())

display(
    spark.table("Bronze_Patient")
    .select(
        "resource_id",
        "resource_type",
        "extraction_timestamp",
        "api_url_or_params"
    )
    .limit(5)
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 14, Finished, Available, Finished, False)

Total Bronze Patient records:
9041


SynapseWidget(Synapse.DataFrame, 22e9a845-c141-4b0c-a9f2-d02831a022cd)

In [13]:

from pyspark.sql import functions as F

resources = ["Encounter", "Observation", "Condition"]

def get_json_files(path):
    files = []
    for item in mssparkutils.fs.ls(path):
        if item.isDir:
            files.extend(get_json_files(item.path))
        elif item.path.endswith(".json"):
            files.append(item.path)
    return files

for resource_name in resources:
    path = f"Files/raw/{resource_name}"
    files = get_json_files(path)

    print(f"{resource_name}: JSON files found = {len(files)}")

    if not files:
        print(f"Skipping {resource_name}: no JSON files found")
        continue

    raw_df = (
        spark.read
        .option("multiLine", "true")
        .json(files)
    )

    bronze_df = (
        raw_df
        .select(
            F.explode_outer("entry").alias("entry"),
            F.current_timestamp().alias("extraction_timestamp")
        )
        .select(
            F.col("entry.fullUrl").alias("full_url"),
            F.col("entry.resource").alias("resource"),
            F.col("extraction_timestamp")
        )
        .filter(F.col("resource").isNotNull())
        .withColumn("resource_id", F.col("resource.id"))
        .withColumn("resource_type", F.col("resource.resourceType"))
        .withColumn(
            "api_url_or_params",
            F.lit(f"https://hapi.fhir.org/baseR4/{resource_name}")
        )
    )

    table_name = f"Bronze_{resource_name}"

    if not spark.catalog.tableExists(table_name):
        bronze_df.write.format("delta").mode("overwrite").saveAsTable(table_name)
        print(f"{table_name} created. Records: {bronze_df.count()}")
    else:
        print(f"{table_name} already exists; skipped to avoid duplicates.")


StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 15, Finished, Available, Finished, False)

Encounter: JSON files found = 107
Bronze_Encounter created. Records: 10603
Observation: JSON files found = 582
Bronze_Observation created. Records: 58122
Condition: JSON files found = 60
Bronze_Condition created. Records: 5944


In [14]:

from pyspark.sql import functions as F

resources = ["Patient", "Encounter", "Observation", "Condition"]

for resource_name in resources:
    bronze_table = f"Bronze_{resource_name}"
    silver_table = f"Silver_{resource_name}"

    df = spark.table(bronze_table)

    # Remove records without a resource ID
    df = df.filter(
        F.col("resource_id").isNotNull()
    )

    # Remove duplicate resource IDs
    df = df.dropDuplicates(["resource_id"])

    # Add processing timestamp
    df = df.withColumn(
        "silver_processed_timestamp",
        F.current_timestamp()
    )

    # Create Silver Delta table
    if not spark.catalog.tableExists(silver_table):
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(silver_table)
        )
        print(
            f"{silver_table} created. "
            f"Records: {spark.table(silver_table).count()}"
        )
    else:
        print(f"{silver_table} already exists; skipped.")


StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 16, Finished, Available, Finished, False)

Silver_Patient created. Records: 9031
Silver_Encounter created. Records: 10603
Silver_Observation created. Records: 58122
Silver_Condition created. Records: 5944


In [16]:

from pyspark.sql import functions as F

resource_tables = [
    ("Patient", "Silver_Patient"),
    ("Encounter", "Silver_Encounter"),
    ("Observation", "Silver_Observation"),
    ("Condition", "Silver_Condition")
]

summary_data = []

for resource_name, table_name in resource_tables:
    total_records = spark.table(table_name).count()

    summary_data.append(
        (resource_name, total_records)
    )

gold_df = spark.createDataFrame(
    summary_data,
    ["resource_name", "total_records"]
).withColumn(
    "report_generated_timestamp",
    F.current_timestamp()
)

gold_df.write.format("delta").mode("overwrite").saveAsTable(
    "Gold_Healthcare_Summary"
)

display(spark.table("Gold_Healthcare_Summary"))


StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c34e8b83-a983-40b1-a9bb-e7003618b4f8)

In [17]:

from pyspark.sql import functions as F

patient_df = spark.table("Silver_Patient")

patient_history_df = (
    patient_df
    .withColumn("is_current", F.lit(True))
    .withColumn("valid_from", F.current_timestamp())
    .withColumn("valid_to", F.lit(None).cast("timestamp"))
    .withColumn(
        "record_hash",
        F.sha2(
            F.to_json(F.col("resource")),
            256
        )
    )
)

if not spark.catalog.tableExists("Gold_Patient_History"):
    (
        patient_history_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("Gold_Patient_History")
    )
    print("Gold_Patient_History created successfully!")
else:
    print("Gold_Patient_History already exists; skipping initialization.")


StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 19, Finished, Available, Finished, False)

Gold_Patient_History created successfully!


In [19]:

display(
    spark.table("Gold_Patient_History")
    .select(
        "resource_id",
        "is_current",
        "valid_from",
        "valid_to",
        "record_hash"
    )
    .limit(5)
)


StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 61e329b3-8e73-4918-aa6d-6e1b54b36c35)

In [ ]:

from delta.tables import DeltaTable
from pyspark.sql import functions as F

# 1. Prepare current source data
source_df = (
    spark.table("Silver_Patient")
    .withColumn(
        "record_hash",
        F.sha2(F.to_json(F.col("resource")), 256)
    )
    .select("resource_id", "resource", "resource_type",
            "extraction_timestamp", "api_url_or_params",
            "record_hash")
    .dropDuplicates(["resource_id"])
)

# 2. Find changed existing Patient records
history_df = spark.table("Gold_Patient_History")

changed_df = (
    source_df.alias("s")
    .join(
        history_df.filter(F.col("is_current") == True).alias("h"),
        "resource_id",
        "inner"
    )
    .filter(F.col("s.record_hash") != F.col("h.record_hash"))
    .select("resource_id")
    .distinct()
)

# 3. Close old versions for changed patients
delta_history = DeltaTable.forName(spark, "Gold_Patient_History")

(
    delta_history.alias("t")
    .merge(
        changed_df.alias("s"),
        "t.resource_id = s.resource_id AND t.is_current = true"
    )
    .whenMatchedUpdate(set={
        "is_current": "false",
        "valid_to": "current_timestamp()"
    })
    .execute()
)

# 4. Insert new versions for changed or new patients
current_history = (
    spark.table("Gold_Patient_History")
    .filter(F.col("is_current") == True)
    .select("resource_id")
    .distinct()
)

new_versions = (
    source_df.alias("s")
    .join(
        current_history.alias("h"),
        "resource_id",
        "left_anti"
    )
    .withColumn("is_current", F.lit(True))
    .withColumn("valid_from", F.current_timestamp())
    .withColumn("valid_to", F.lit(None).cast("timestamp"))
)

(
    new_versions.write
    .format("delta")
    .mode("append")
    .saveAsTable("Gold_Patient_History")
)

print("SCD Type 2 processing completed.")
print("Changed Patient IDs:", changed_df.count())
print("Current Patient versions:",
      spark.table("Gold_Patient_History")
      .filter(F.col("is_current") == True).count())


In [20]:

from pyspark.sql import functions as F

history_df = spark.table("Gold_Patient_History")

print("Total history records:", history_df.count())

print("Current records:")
print(
    history_df.filter(F.col("is_current") == True).count()
)

print("Historical records:")
print(
    history_df.filter(F.col("is_current") == False).count()
)

display(
    history_df.select(
        "resource_id",
        "is_current",
        "valid_from",
        "valid_to"
    ).limit(10)
)


StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 22, Finished, Available, Finished, False)

Total history records: 9031
Current records:
9031
Historical records:
0


SynapseWidget(Synapse.DataFrame, da82dd62-4bba-4bdb-896a-a2928c9ee18b)

In [21]:

# Create an isolated test copy
(
    spark.table("Gold_Patient_History")
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("Test_Patient_History")
)

print("Test_Patient_History created.")


StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 23, Finished, Available, Finished, False)

Test_Patient_History created.


In [22]:

from pyspark.sql import functions as F

test_df = spark.table("Test_Patient_History")

# Select one current Patient for testing
test_patient = (
    test_df
    .filter(F.col("is_current") == True)
    .select("resource_id")
    .first()
)

test_id = test_patient["resource_id"]
print("Selected test Patient ID:", test_id)

# Change the hash for this test record to simulate a source change
(
    test_df
    .withColumn(
        "record_hash",
        F.when(
            (F.col("resource_id") == test_id) &
            (F.col("is_current") == True),
            F.sha2(
                F.concat(
                    F.coalesce(F.col("record_hash"), F.lit("")),
                    F.lit("_TEST_CHANGE")
                ),
                256
            )
        ).otherwise(F.col("record_hash"))
    )
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("Test_Patient_History")
)

print("Test change simulated successfully.")


StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 24, Finished, Available, Finished, False)

Selected test Patient ID: 045ed820-8c47-477d-a6d9-25b69e411b7d
Test change simulated successfully.


In [23]:

from delta.tables import DeltaTable
from pyspark.sql import functions as F

table_name = "Test_Patient_History"

# Find the Patient whose hash was changed
current_df = spark.table(table_name).filter(
    F.col("is_current") == True
)

changed_patient = (
    current_df
    .filter(F.col("record_hash").endswith(
        F.lit("_TEST_CHANGE")
    ))
    .select("resource_id")
    .first()
)

# The hash was replaced with SHA-256, so identify the test ID directly
test_id = "045ed820-8c47-477d-a6d9-25b69e411b7d"

# Close the existing version
delta_table = DeltaTable.forName(spark, table_name)

(
    delta_table.alias("t")
    .merge(
        spark.createDataFrame([(test_id,)], ["resource_id"]).alias("s"),
        "t.resource_id = s.resource_id AND t.is_current = true"
    )
    .whenMatchedUpdate(set={
        "is_current": "false",
        "valid_to": "current_timestamp()"
    })
    .execute()
)

# Insert a new current version with the changed hash
old_version = (
    spark.table(table_name)
    .filter(F.col("resource_id") == test_id)
    .orderBy(F.col("valid_from").desc())
    .first()
)

new_version = (
    spark.table(table_name)
    .filter(
        (F.col("resource_id") == test_id) &
        (F.col("is_current") == False)
    )
    .orderBy(F.col("valid_from").desc())
    .limit(1)
    .withColumn("is_current", F.lit(True))
    .withColumn("valid_from", F.current_timestamp())
    .withColumn("valid_to", F.lit(None).cast("timestamp"))
)

new_version.write.format("delta").mode("append").saveAsTable(table_name)

print("SCD Type 2 test executed.")
display(
    spark.table(table_name)
    .filter(F.col("resource_id") == test_id)
    .select("resource_id", "is_current", "valid_from", "valid_to", "record_hash")
)


StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 25, Finished, Available, Finished, False)

SCD Type 2 test executed.


SynapseWidget(Synapse.DataFrame, 79e5129e-3ebf-46a5-9053-f58ff24435f6)

In [24]:

test_id = "045ed820-8c47-477d-a6d9-25b69e411b7d"

display(
    spark.table("Test_Patient_History")
    .filter(F.col("resource_id") == test_id)
    .select(
        "resource_id",
        "is_current",
        "valid_from",
        "valid_to",
        "record_hash"
    )
    .orderBy("valid_from")
)


StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 26, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3954409f-221a-4ec5-bb81-0314fa5e567b)

In [25]:

from pyspark.sql import functions as F

history_df = spark.table("Gold_Patient_History")

print("Total history records:", history_df.count())

history_df.printSchema()

display(
    history_df
    .select(
        "resource_id",
        "resource_type",
        "is_current",
        "valid_from",
        "valid_to",
        "record_hash"
    )
    .limit(5)
)


StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 27, Finished, Available, Finished, False)

Total history records: 9031
root
 |-- full_url: string (nullable = true)
 |-- resource: struct (nullable = true)
 |    |-- _birthDate: struct (nullable = true)
 |    |    |-- extension: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- url: string (nullable = true)
 |    |    |    |    |-- valueDateTime: string (nullable = true)
 |    |-- _gender: struct (nullable = true)
 |    |    |-- extension: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- url: string (nullable = true)
 |    |    |    |    |-- valueCode: string (nullable = true)
 |    |    |    |    |-- valueCodeableConcept: struct (nullable = true)
 |    |    |    |    |    |-- coding: array (nullable = true)
 |    |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |    |-- code: string (nullable = true)
 |    |    |    |    |    |    |    |-- display: string (nullable 

SynapseWidget(Synapse.DataFrame, 92e0bf34-37bc-44f5-877b-02df2b34f4a7)

In [26]:

from pyspark.sql import functions as F

# Current Patient source
source_df = (
    spark.table("Silver_Patient")
    .withColumn(
        "new_record_hash",
        F.sha2(F.to_json(F.col("resource")), 256)
    )
    .select(
        "resource_id",
        "resource",
        "resource_type",
        "extraction_timestamp",
        "api_url_or_params",
        "new_record_hash"
    )
    .dropDuplicates(["resource_id"])
)

# Existing current history versions
history_current_df = (
    spark.table("Gold_Patient_History")
    .filter(F.col("is_current") == True)
    .select("resource_id", "record_hash")
)

# Detect changed and new Patient records
comparison_df = (
    source_df.alias("s")
    .join(
        history_current_df.alias("h"),
        on="resource_id",
        how="left"
    )
    .withColumn(
        "change_status",
        F.when(
            F.col("h.record_hash").isNull(),
            F.lit("NEW")
        ).when(
            F.col("s.new_record_hash") != F.col("h.record_hash"),
            F.lit("CHANGED")
        ).otherwise(F.lit("UNCHANGED"))
    )
)

display(
    comparison_df
    .groupBy("change_status")
    .count()
)


StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 28, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b4a1dde4-4ba0-48d1-878a-3b7f3b308a67)

In [27]:

from pyspark.sql import functions as F

# Select one Patient for an isolated change-detection test
test_patient_id = (
    spark.table("Silver_Patient")
    .select("resource_id")
    .first()["resource_id"]
)

print("Test Patient ID:", test_patient_id)

test_source_df = (
    spark.table("Silver_Patient")
    .withColumn(
        "new_record_hash",
        F.sha2(F.to_json(F.col("resource")), 256)
    )
    .withColumn(
        "test_resource",
        F.when(
            F.col("resource_id") == test_patient_id,
            F.to_json(
                F.col("resource")
            )
        ).otherwise(F.to_json(F.col("resource")))
    )
)

display(
    test_source_df
    .filter(F.col("resource_id") == test_patient_id)
    .select("resource_id", "new_record_hash", "test_resource")
)


StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 29, Finished, Available, Finished, False)

Test Patient ID: 018f428e-34f6-4707-8009-5ad742f901e7


SynapseWidget(Synapse.DataFrame, 22cbbf88-e224-45d4-ac51-2e1de7f814d5)

In [28]:

from pyspark.sql import functions as F

test_patient_id = "018f428e-34f6-4707-8009-5ad742f901e7"

test_source = (
    spark.table("Silver_Patient")
    .withColumn(
        "test_resource",
        F.when(
            F.col("resource_id") == test_patient_id,
            F.to_json(
                F.col("resource").withField(
                    "gender", F.lit("female")
                )
            )
        ).otherwise(F.to_json(F.col("resource")))
    )
    .withColumn(
        "test_hash",
        F.sha2(F.col("test_resource"), 256)
    )
)

display(
    test_source
    .filter(F.col("resource_id") == test_patient_id)
    .select("resource_id", "test_resource", "test_hash")
)


StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 30, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fdc9beb8-1ca2-4b0c-962a-553891282ca9)

In [29]:

from pyspark.sql import functions as F
from delta.tables import DeltaTable

test_id = "018f428e-34f6-4707-8009-5ad742f901e7"

# Use the isolated test history table
test_history = DeltaTable.forName(spark, "Test_Patient_History")

# Prepare the changed source record
changed_source = (
    spark.table("Silver_Patient")
    .filter(F.col("resource_id") == test_id)
    .withColumn(
        "resource",
        F.col("resource").withField("gender", F.lit("female"))
    )
    .withColumn(
        "record_hash",
        F.sha2(F.to_json(F.col("resource")), 256)
    )
    .withColumn("is_current", F.lit(True))
    .withColumn("valid_from", F.current_timestamp())
    .withColumn("valid_to", F.lit(None).cast("timestamp"))
)

# Close the existing current version only when the hash differs
(
    test_history.alias("t")
    .merge(
        changed_source.select("resource_id", "record_hash").alias("s"),
        "t.resource_id = s.resource_id AND t.is_current = true"
    )
    .whenMatchedUpdate(
        condition="t.record_hash <> s.record_hash",
        set={
            "is_current": "false",
            "valid_to": "current_timestamp()"
        }
    )
    .execute()
)

# Insert the new changed version if no current version remains
current_ids = (
    spark.table("Test_Patient_History")
    .filter(F.col("is_current") == True)
    .select("resource_id")
    .distinct()
)

new_versions = (
    changed_source
    .join(current_ids, "resource_id", "left_anti")
    .select(
        "resource_id",
        "resource_type",
        "resource",
        "extraction_timestamp",
        "api_url_or_params",
        "record_hash",
        "is_current",
        "valid_from",
        "valid_to"
    )
)

new_versions.write.format("delta").mode("append").saveAsTable(
    "Test_Patient_History"
)

print("Test SCD Type 2 processing complete.")

display(
    spark.table("Test_Patient_History")
    .filter(F.col("resource_id") == test_id)
    .select(
        "resource_id",
        "resource.gender",
        "record_hash",
        "is_current",
        "valid_from",
        "valid_to"
    )
    .orderBy("valid_from")
)


StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 31, Finished, Available, Finished, False)

Test SCD Type 2 processing complete.


SynapseWidget(Synapse.DataFrame, 1560a123-80d8-4b62-a050-63112805ca21)

In [30]:
from pyspark.sql import functions as F

test_id = "018f428e-34f6-4707-8009-5ad742f901e7"

display(
    spark.table("Test_Patient_History")
    .filter(F.col("resource_id") == test_id)
    .groupBy("resource_id")
    .agg(
        F.count("*").alias("total_versions"),
        F.sum(
            F.when(F.col("is_current") == True, 1).otherwise(0)
        ).alias("current_versions"),
        F.sum(
            F.when(F.col("is_current") == False, 1).otherwise(0)
        ).alias("historical_versions")
    )
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 32, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b2a1b636-0fb8-492c-8eb1-734c8f980123)

In [31]:
from pyspark.sql import functions as F

test_id = "018f428e-34f6-4707-8009-5ad742f901e7"

display(
    spark.table("Test_Patient_History")
    .filter(F.col("resource_id") == test_id)
    .groupBy("resource_id")
    .agg(
        F.count("*").alias("total_versions"),
        F.sum(
            F.when(F.col("is_current") == True, 1).otherwise(0)
        ).alias("current_versions"),
        F.sum(
            F.when(F.col("is_current") == False, 1).otherwise(0)
        ).alias("historical_versions")
    )
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 33, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6bb7940f-cbd0-4597-a6ce-8e05ce4708d6)

In [32]:
from pyspark.sql import functions as F

silver = spark.table("Silver_Patient")
history = spark.table("Gold_Patient_History")

current_history = history.filter(F.col("is_current") == True)

print("Silver Patient count:", silver.count())
print("History total versions:", history.count())
print("Current history records:", current_history.count())

print("Duplicate current Patient IDs:")
display(
    current_history
    .groupBy("resource_id")
    .count()
    .filter(F.col("count") > 1)
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 34, Finished, Available, Finished, False)

Silver Patient count: 9031
History total versions: 9031
Current history records: 9031
Duplicate current Patient IDs:


SynapseWidget(Synapse.DataFrame, 68b7d659-cf81-463c-af3a-8d135f9498fe)

In [33]:
from pyspark.sql import functions as F

silver = (
    spark.table("Silver_Patient")
    .withColumn(
        "source_hash",
        F.sha2(F.to_json(F.col("resource")), 256)
    )
    .select("resource_id", "source_hash")
)

current_history = (
    spark.table("Gold_Patient_History")
    .filter(F.col("is_current") == True)
    .select("resource_id", "record_hash")
)

comparison = (
    silver.alias("s")
    .join(current_history.alias("h"), "resource_id", "full")
    .withColumn(
        "status",
        F.when(F.col("h.record_hash").isNull(), "NEW")
         .when(F.col("s.source_hash").isNull(), "MISSING_IN_SOURCE")
         .when(
             F.col("source_hash") != F.col("record_hash"),
             "CHANGED"
         )
         .otherwise("UNCHANGED")
    )
)

display(comparison.groupBy("status").count())

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 35, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4dc492a4-726b-490b-946e-2c6823f11813)

In [34]:
from pyspark.sql import functions as F

silver_ids = (
    spark.table("Silver_Patient")
    .select("resource_id")
    .distinct()
)

history_ids = (
    spark.table("Gold_Patient_History")
    .filter(F.col("is_current") == True)
    .select("resource_id")
    .distinct()
)

new_patients = silver_ids.join(
    history_ids,
    on="resource_id",
    how="left_anti"
)

print("New Patient IDs:", new_patients.count())
display(new_patients.limit(10))

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 36, Finished, Available, Finished, False)

New Patient IDs: 0


SynapseWidget(Synapse.DataFrame, 9c281a33-dbeb-4dbc-98f5-9c646acc64cf)

In [35]:
spark.table("Gold_Patient_History").printSchema()

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 37, Finished, Available, Finished, False)

root
 |-- full_url: string (nullable = true)
 |-- resource: struct (nullable = true)
 |    |-- _birthDate: struct (nullable = true)
 |    |    |-- extension: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- url: string (nullable = true)
 |    |    |    |    |-- valueDateTime: string (nullable = true)
 |    |-- _gender: struct (nullable = true)
 |    |    |-- extension: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- url: string (nullable = true)
 |    |    |    |    |-- valueCode: string (nullable = true)
 |    |    |    |    |-- valueCodeableConcept: struct (nullable = true)
 |    |    |    |    |    |-- coding: array (nullable = true)
 |    |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |    |-- code: string (nullable = true)
 |    |    |    |    |    |    |    |-- display: string (nullable = true)
 |    |    |    |   

In [36]:
from pyspark.sql import functions as F

# Source records with a hash of the complete FHIR resource
source = (
    spark.table("Silver_Patient")
    .withColumn(
        "record_hash",
        F.sha2(F.to_json(F.col("resource")), 256)
    )
    .select(
        "resource_id",
        "record_hash"
    )
)

# Only the current production history versions
current = (
    spark.table("Gold_Patient_History")
    .filter(F.col("is_current") == True)
    .select(
        "resource_id",
        F.col("record_hash").alias("history_hash")
    )
)

comparison = (
    source.alias("s")
    .join(current.alias("h"), "resource_id", "full_outer")
    .withColumn(
        "status",
        F.when(F.col("history_hash").isNull(), "NEW")
         .when(F.col("record_hash").isNull(), "MISSING_IN_SOURCE")
         .when(
             F.col("record_hash") != F.col("history_hash"),
             "CHANGED"
         )
         .otherwise("UNCHANGED")
    )
)

display(comparison.groupBy("status").count())

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 38, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0c692442-419c-4512-aaeb-7880c4be30f3)

In [37]:
backup_table = "Gold_Patient_History_Backup"

spark.table("Gold_Patient_History") \
    .write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(backup_table)

print("Backup created:", backup_table)
print("Original rows:", spark.table("Gold_Patient_History").count())
print("Backup rows:", spark.table(backup_table).count())

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 39, Finished, Available, Finished, False)

Backup created: Gold_Patient_History_Backup
Original rows: 9031
Backup rows: 9031


In [38]:
backup = spark.table("Gold_Patient_History_Backup")

print("Backup rows:", backup.count())
print("Backup current rows:", backup.filter("is_current = true").count())
print("Backup historical rows:", backup.filter("is_current = false").count())

backup.printSchema()

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 40, Finished, Available, Finished, False)

Backup rows: 9031
Backup current rows: 9031
Backup historical rows: 0
root
 |-- full_url: string (nullable = true)
 |-- resource: struct (nullable = true)
 |    |-- _birthDate: struct (nullable = true)
 |    |    |-- extension: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- url: string (nullable = true)
 |    |    |    |    |-- valueDateTime: string (nullable = true)
 |    |-- _gender: struct (nullable = true)
 |    |    |-- extension: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- url: string (nullable = true)
 |    |    |    |    |-- valueCode: string (nullable = true)
 |    |    |    |    |-- valueCodeableConcept: struct (nullable = true)
 |    |    |    |    |    |-- coding: array (nullable = true)
 |    |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |    |-- code: string (nullable = true)
 |    |    |    |    |  

In [39]:
from pyspark.sql import functions as F

test_id = "018f428e-34f6-4707-8009-5ad742f901e7"

display(
    spark.table("Test_Patient_History")
    .filter(F.col("resource_id") == test_id)
    .select(
        "resource_id",
        "resource.gender",
        "is_current",
        "valid_from",
        "valid_to",
        "record_hash"
    )
    .orderBy("valid_from")
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 41, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d1f0c134-c137-46b9-bbb5-b674c933f07d)

In [40]:
spark.table("Test_Patient_History") \
    .write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("Test_Patient_History_Rerun")

print(
    "Test rows:",
    spark.table("Test_Patient_History_Rerun").count()
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 42, Finished, Available, Finished, False)

Test rows: 9033


In [41]:
from pyspark.sql import functions as F

test_history = spark.table("Test_Patient_History_Rerun")

print("Total versions:", test_history.count())
print(
    "Current versions:",
    test_history.filter(F.col("is_current") == True).count()
)
print(
    "Historical versions:",
    test_history.filter(F.col("is_current") == False).count()
)

print("Duplicate current Patient IDs:")
display(
    test_history
    .filter(F.col("is_current") == True)
    .groupBy("resource_id")
    .count()
    .filter(F.col("count") > 1)
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 43, Finished, Available, Finished, False)

Total versions: 9033
Current versions: 9031
Historical versions: 2
Duplicate current Patient IDs:


SynapseWidget(Synapse.DataFrame, 89376080-cf56-4b7f-987b-a0181b59e759)

In [42]:
from pyspark.sql import functions as F

source = (
    spark.table("Silver_Patient")
    .withColumn(
        "source_hash",
        F.sha2(F.to_json(F.col("resource")), 256)
    )
    .select("resource_id", "source_hash")
)

current = (
    spark.table("Test_Patient_History_Rerun")
    .filter(F.col("is_current") == True)
    .select(
        "resource_id",
        F.col("record_hash").alias("history_hash")
    )
)

result = (
    source.alias("s")
    .join(current.alias("h"), "resource_id", "full_outer")
    .withColumn(
        "status",
        F.when(F.col("history_hash").isNull(), "NEW")
         .when(F.col("source_hash").isNull(), "MISSING_IN_SOURCE")
         .when(
             F.col("source_hash") != F.col("history_hash"),
             "CHANGED"
         )
         .otherwise("UNCHANGED")
    )
)

display(result.groupBy("status").count())

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 44, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e67e5957-80af-476c-aaea-57b1469589ce)

In [43]:
from pyspark.sql import functions as F

source = (
    spark.table("Silver_Patient")
    .withColumn(
        "source_hash",
        F.sha2(F.to_json(F.col("resource")), 256)
    )
    .select(
        "resource_id",
        F.col("resource.gender").alias("source_gender"),
        "source_hash"
    )
)

current = (
    spark.table("Test_Patient_History_Rerun")
    .filter(F.col("is_current") == True)
    .select(
        "resource_id",
        F.col("resource.gender").alias("history_gender"),
        F.col("record_hash").alias("history_hash")
    )
)

display(
    source.join(current, "resource_id", "inner")
    .filter(F.col("source_hash") != F.col("history_hash"))
    .select(
        "resource_id",
        "source_gender",
        "history_gender",
        "source_hash",
        "history_hash"
    )
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 45, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e69677f4-cec3-48ed-9cb1-c9ec85765620)

In [44]:
from pyspark.sql import functions as F

check_ids = [
    "018f428e-34f6-4707-8009-5ad742f901e7",
    "045ed820-8c47-477d-a6d9-25b69e411b7d"
]

display(
    spark.table("Test_Patient_History_Rerun")
    .filter(F.col("resource_id").isin(check_ids))
    .select(
        "resource_id",
        F.col("resource.gender").alias("gender"),
        "record_hash",
        "is_current",
        "valid_from",
        "valid_to"
    )
    .orderBy("resource_id", "valid_from")
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 46, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5f84dabd-b3ad-46ff-83cc-7f52eb308fd2)

In [45]:
spark.sql("""
CREATE OR REPLACE TABLE Test_Patient_History_Clean
USING DELTA
AS
SELECT *
FROM Gold_Patient_History
""")

clean_table = spark.table("Test_Patient_History_Clean")

print("Total records:", clean_table.count())
print(
    "Current records:",
    clean_table.filter("is_current = true").count()
)
print(
    "Historical records:",
    clean_table.filter("is_current = false").count()
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 47, Finished, Available, Finished, False)

Total records: 9031
Current records: 9031
Historical records: 0


In [46]:
from pyspark.sql import functions as F

source = (
    spark.table("Silver_Patient")
    .withColumn(
        "source_hash",
        F.sha2(F.to_json(F.col("resource")), 256)
    )
    .select("resource_id", "source_hash")
)

current = (
    spark.table("Test_Patient_History_Clean")
    .filter(F.col("is_current") == True)
    .select(
        "resource_id",
        F.col("record_hash").alias("history_hash")
    )
)

comparison = (
    source.alias("s")
    .join(current.alias("h"), "resource_id", "full_outer")
    .withColumn(
        "status",
        F.when(F.col("history_hash").isNull(), "NEW")
         .when(F.col("source_hash").isNull(), "MISSING_IN_SOURCE")
         .when(
             F.col("source_hash") != F.col("history_hash"),
             "CHANGED"
         )
         .otherwise("UNCHANGED")
    )
)

display(comparison.groupBy("status").count())

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 48, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 332e6bae-fa98-44e8-8390-d1a92de2d089)

In [47]:
from pyspark.sql import functions as F

test_id = "018f428e-34f6-4707-8009-5ad742f901e7"

display(
    spark.table("Test_Patient_History_Clean")
    .filter(F.col("resource_id") == test_id)
    .select(
        "resource_id",
        F.col("resource.gender").alias("gender"),
        "record_hash",
        "is_current",
        "valid_from",
        "valid_to"
    )
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 49, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1aab6ff7-c62d-4460-8f4d-507d8103d7f4)

In [48]:
from pyspark.sql import functions as F

test_id = "018f428e-34f6-4707-8009-5ad742f901e7"

# Read the current test record
test_df = (
    spark.table("Test_Patient_History_Clean")
    .filter(F.col("resource_id") == test_id)
)

# Simulate a source change: male -> female
changed_resource = F.to_json(
    F.col("resource"),
    options={"ignoreNullFields": "false"}
)

synthetic_change = (
    test_df
    .withColumn(
        "resource",
        F.from_json(
            F.to_json(
                F.col("resource")
            ),
            test_df.schema["resource"].dataType
        )
    )
    .withColumn(
        "resource",
        F.expr(
            "resource"
        )
    )
)

display(
    test_df.select(
        "resource_id",
        F.col("resource.gender").alias("current_gender"),
        "is_current"
    )
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 50, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9384d9ed-cdc7-4c8c-a712-915c9c015b12)

In [49]:
from pyspark.sql import functions as F

test_id = "018f428e-34f6-4707-8009-5ad742f901e7"

# Read selected current record
df = spark.table("Test_Patient_History_Clean")

# Create a modified resource with gender = female
changed_df = (
    df.filter(
        (F.col("resource_id") == test_id) &
        (F.col("is_current") == True)
    )
    .withColumn(
        "resource",
        F.expr(
            "replace(to_json(resource), '\"gender\":\"male\"', '\"gender\":\"female\"')"
        )
    )
)

display(changed_df.select("resource_id", "resource"))

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 51, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3ca9a3f3-d235-4c0a-85e5-20ad49f70db7)

In [50]:
from pyspark.sql import functions as F

test_id = "018f428e-34f6-4707-8009-5ad742f901e7"

df = spark.table("Test_Patient_History_Clean")
resource_schema = df.schema["resource"].dataType

changed_record = (
    df.filter(
        (F.col("resource_id") == test_id) &
        (F.col("is_current") == True)
    )
    .withColumn(
        "resource_json",
        F.regexp_replace(
            F.to_json("resource"),
            '"gender":"male"',
            '"gender":"female"'
        )
    )
    .withColumn(
        "resource",
        F.from_json("resource_json", resource_schema)
    )
    .drop("resource_json")
    .withColumn(
        "record_hash",
        F.sha2(F.to_json("resource"), 256)
    )
)

display(
    changed_record.select(
        "resource_id",
        F.col("resource.gender").alias("simulated_gender"),
        "record_hash",
        "is_current"
    )
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 52, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ba6db99c-4a29-4452-ab14-5801a49f4651)

In [51]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

table_name = "Test_Patient_History_Clean"
test_id = "018f428e-34f6-4707-8009-5ad742f901e7"

# Prepare the simulated changed record
df = spark.table(table_name)
resource_schema = df.schema["resource"].dataType

changed_record = (
    df.filter(
        (F.col("resource_id") == test_id) &
        (F.col("is_current") == True)
    )
    .withColumn(
        "resource_json",
        F.regexp_replace(
            F.to_json("resource"),
            '"gender":"male"',
            '"gender":"female"'
        )
    )
    .withColumn(
        "resource",
        F.from_json("resource_json", resource_schema)
    )
    .drop("resource_json")
    .withColumn(
        "record_hash",
        F.sha2(F.to_json("resource"), 256)
    )
)

new_hash = changed_record.select("record_hash").first()["record_hash"]

# Close old current version only if the hash differs
DeltaTable.forName(spark, table_name).update(
    condition=(
        (F.col("resource_id") == test_id) &
        (F.col("is_current") == True) &
        (F.col("record_hash") != F.lit(new_hash))
    ),
    set={
        "is_current": "false",
        "valid_to": "current_timestamp()"
    }
)

# Insert the new version only if it does not already exist as current
current_exists = (
    spark.table(table_name)
    .filter(
        (F.col("resource_id") == test_id) &
        (F.col("is_current") == True) &
        (F.col("record_hash") == new_hash)
    )
    .limit(1)
    .count()
)

if current_exists == 0:
    (
        changed_record
        .withColumn("is_current", F.lit(True))
        .withColumn("valid_from", F.current_timestamp())
        .withColumn("valid_to", F.lit(None).cast("timestamp"))
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(table_name)
    )

print("SCD Type 2 operation completed.")

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 53, Finished, Available, Finished, False)

SCD Type 2 operation completed.


In [52]:
display(
    spark.table("Test_Patient_History_Clean")
    .filter(F.col("resource_id") == test_id)
    .select(
        "resource_id",
        F.col("resource.gender").alias("gender"),
        "record_hash",
        "is_current",
        "valid_from",
        "valid_to"
    )
    .orderBy("valid_from")
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 54, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5fe1fb38-d7d0-4304-9788-c13660b7e7e2)

In [53]:
from pyspark.sql import functions as F

test_id = "018f428e-34f6-4707-8009-5ad742f901e7"

display(
    spark.table("Test_Patient_History_Clean")
    .filter(F.col("resource_id") == test_id)
    .select(
        "resource_id",
        F.col("resource.gender").alias("gender"),
        "record_hash",
        "is_current",
        "valid_from",
        "valid_to"
    )
    .orderBy("valid_from")
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 55, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ddf97a02-1017-46aa-8004-d604df54915d)

In [54]:
from pyspark.sql import functions as F

test_id = "018f428e-34f6-4707-8009-5ad742f901e7"

display(
    spark.table("Test_Patient_History_Clean")
    .filter(F.col("resource_id") == test_id)
    .select(
        "resource_id",
        F.col("resource.gender").alias("gender"),
        "record_hash",
        "is_current",
        "valid_from",
        "valid_to"
    )
)

print(
    "Current version count:",
    spark.table("Test_Patient_History_Clean")
    .filter(
        (F.col("resource_id") == test_id) &
        (F.col("is_current") == True)
    )
    .count()
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 56, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 835419c4-0ae1-4d71-bcec-881ed63285e2)

Current version count: 0


In [55]:
from pyspark.sql import functions as F

table_name = "Test_Patient_History_Clean"
test_id = "018f428e-34f6-4707-8009-5ad742f901e7"

df = spark.table(table_name)
resource_schema = df.schema["resource"].dataType

# Get the closed historical version as the base record
old_record = (
    df.filter(
        (F.col("resource_id") == test_id) &
        (F.col("is_current") == False)
    )
    .orderBy(F.col("valid_from").desc())
    .limit(1)
)

# Change gender in the JSON and parse back to the original FHIR struct
new_record = (
    old_record
    .withColumn(
        "resource_json",
        F.regexp_replace(
            F.to_json("resource"),
            '"gender":"male"',
            '"gender":"female"'
        )
    )
    .withColumn(
        "resource",
        F.from_json("resource_json", resource_schema)
    )
    .drop("resource_json")
    .withColumn(
        "record_hash",
        F.sha2(F.to_json("resource"), 256)
    )
    .withColumn("is_current", F.lit(True))
    .withColumn("valid_from", F.current_timestamp())
    .withColumn("valid_to", F.lit(None).cast("timestamp"))
)

# Insert only if the new current version is not already present
already_exists = (
    spark.table(table_name)
    .filter(
        (F.col("resource_id") == test_id) &
        (F.col("is_current") == True) &
        (F.col("resource.gender") == "female")
    )
    .limit(1)
    .count()
)

if already_exists == 0:
    new_record.write.format("delta").mode("append").saveAsTable(table_name)
    print("New female version inserted.")
else:
    print("Female current version already exists; no insert needed.")

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 57, Finished, Available, Finished, False)

New female version inserted.


In [56]:
display(
    spark.table("Test_Patient_History_Clean")
    .filter(F.col("resource_id") == test_id)
    .select(
        "resource_id",
        F.col("resource.gender").alias("gender"),
        "record_hash",
        "is_current",
        "valid_from",
        "valid_to"
    )
    .orderBy("valid_from")
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 58, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b66c05e6-0365-411d-a28e-2dd29b9e8a85)

In [57]:
from pyspark.sql import functions as F

test_id = "018f428e-34f6-4707-8009-5ad742f901e7"

patient_history = (
    spark.table("Test_Patient_History_Clean")
    .filter(F.col("resource_id") == test_id)
)

total_versions = patient_history.count()

current_versions = patient_history.filter(
    F.col("is_current") == True
).count()

historical_versions = patient_history.filter(
    F.col("is_current") == False
).count()

print("Total versions:", total_versions)
print("Current versions:", current_versions)
print("Historical versions:", historical_versions)

assert total_versions == 2, "Unexpected total versions"
assert current_versions == 1, "Expected exactly one current version"
assert historical_versions == 1, "Expected exactly one historical version"

print("SCD Type 2 validation PASSED")

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 59, Finished, Available, Finished, False)

Total versions: 2
Current versions: 1
Historical versions: 1
SCD Type 2 validation PASSED


In [58]:
from pyspark.sql import functions as F

source = (
    spark.table("Silver_Patient")
    .withColumn(
        "source_hash",
        F.sha2(F.to_json(F.col("resource")), 256)
    )
    .select("resource_id", "source_hash")
)

current = (
    spark.table("Test_Patient_History_Clean")
    .filter(F.col("is_current") == True)
    .select(
        "resource_id",
        F.col("record_hash").alias("history_hash")
    )
)

comparison = (
    source.alias("s")
    .join(current.alias("h"), "resource_id", "full_outer")
    .withColumn(
        "status",
        F.when(F.col("history_hash").isNull(), "NEW")
         .when(F.col("source_hash").isNull(), "MISSING_IN_SOURCE")
         .when(
             F.col("source_hash") != F.col("history_hash"),
             "CHANGED"
         )
         .otherwise("UNCHANGED")
    )
)

display(comparison.groupBy("status").count())

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 60, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d4d5b6bd-53a2-451e-a11c-02c05df02f71)

In [59]:
from pyspark.sql import functions as F

table_name = "Test_Patient_History_Clean"
test_id = "018f428e-34f6-4707-8009-5ad742f901e7"

patient_history = (
    spark.table(table_name)
    .filter(F.col("resource_id") == test_id)
)

total_versions = patient_history.count()

current_versions = patient_history.filter(
    F.col("is_current") == True
).count()

historical_versions = patient_history.filter(
    F.col("is_current") == False
).count()

duplicate_current_ids = (
    spark.table(table_name)
    .filter(F.col("is_current") == True)
    .groupBy("resource_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Total versions for test patient:", total_versions)
print("Current versions:", current_versions)
print("Historical versions:", historical_versions)
print("Duplicate current patient IDs in table:", duplicate_current_ids)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 61, Finished, Available, Finished, False)

Total versions for test patient: 2
Current versions: 1
Historical versions: 1
Duplicate current patient IDs in table: 0


In [60]:
from pyspark.sql import functions as F

table_name = "Test_Patient_History_Clean"

current_snapshot = (
    spark.table(table_name)
    .filter(F.col("is_current") == True)
    .select("resource_id", "record_hash")
)

print("Current records:", current_snapshot.count())
print(
    "Duplicate current IDs:",
    current_snapshot.groupBy("resource_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 62, Finished, Available, Finished, False)

Current records: 9031
Duplicate current IDs: 0


In [61]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

def apply_scd2(source_table, target_table):
    source = (
        spark.table(source_table)
        .withColumn(
            "source_hash",
            F.sha2(F.to_json(F.col("resource")), 256)
        )
    )

    target = spark.table(target_table)

    current = (
        target
        .filter(F.col("is_current") == True)
        .select(
            "resource_id",
            F.col("record_hash").alias("target_hash")
        )
    )

    comparison = (
        source.alias("s")
        .join(current.alias("t"), "resource_id", "left")
        .withColumn(
            "status",
            F.when(F.col("target_hash").isNull(), "NEW")
             .when(
                 F.col("source_hash") != F.col("target_hash"),
                 "CHANGED"
             )
             .otherwise("UNCHANGED")
        )
    )

    print("Source comparison:")
    display(comparison.groupBy("status").count())

    print("Function prepared. No target records were modified.")

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 63, Finished, Available, Finished, False)

In [62]:
apply_scd2(
    source_table="Silver_Patient",
    target_table="Test_Patient_History_Clean"
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 64, Finished, Available, Finished, False)

Source comparison:


SynapseWidget(Synapse.DataFrame, 296f77ac-4bdd-46d4-a6c0-98598e42c35f)

Function prepared. No target records were modified.


In [65]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

def apply_scd2(source_table, target_table):
    source_df = (
        spark.table(source_table)
        .withColumn(
            "source_hash",
            F.sha2(F.to_json(F.col("resource")), 256)
        )
    )

    target_df = spark.table(target_table)

    current_df = (
        target_df
        .filter(F.col("is_current") == True)
        .select(
            "resource_id",
            F.col("record_hash").alias("target_hash")
        )
    )

    comparison = (
        source_df.alias("s")
        .join(current_df.alias("t"), "resource_id", "left")
        .withColumn(
            "status",
            F.when(F.col("target_hash").isNull(), "NEW")
             .when(
                 F.col("source_hash") != F.col("target_hash"),
                 "CHANGED"
             )
             .otherwise("UNCHANGED")
        )
    )

    changed_ids = (
        comparison
        .filter(F.col("status") == "CHANGED")
        .select("resource_id")
        .distinct()
    )

    new_ids = (
        comparison
        .filter(F.col("status") == "NEW")
        .select("resource_id")
        .distinct()
    )

    print("Changed IDs:", changed_ids.count())
    print("New IDs:", new_ids.count())
    print("Function updated; no table records modified yet.")

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 67, Finished, Available, Finished, False)

In [66]:
print("Notebook is working")

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 68, Finished, Available, Finished, False)

Notebook is working


In [67]:
from pyspark.sql import functions as F

source = (
    spark.table("Silver_Patient")
    .withColumn(
        "source_hash",
        F.sha2(F.to_json(F.col("resource")), 256)
    )
    .select("resource_id", "source_hash")
)

current = (
    spark.table("Test_Patient_History_Clean")
    .filter(F.col("is_current") == True)
    .select(
        "resource_id",
        F.col("record_hash").alias("target_hash")
    )
)

comparison = (
    source.alias("s")
    .join(current.alias("t"), "resource_id", "left")
    .withColumn(
        "status",
        F.when(F.col("target_hash").isNull(), "NEW")
         .when(
             F.col("source_hash") != F.col("target_hash"),
             "CHANGED"
         )
         .otherwise("UNCHANGED")
    )
)

print("Changed IDs:", comparison.filter(F.col("status") == "CHANGED").count())
print("New IDs:", comparison.filter(F.col("status") == "NEW").count())
print("Comparison completed.")

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 69, Finished, Available, Finished, False)

Changed IDs: 1
New IDs: 0
Comparison completed.


In [68]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

table_name = "Test_Patient_History_Clean"
test_id = "018f428e-34f6-4707-8009-5ad742f901e7"

source_hash = (
    spark.table("Silver_Patient")
    .filter(F.col("resource_id") == test_id)
    .withColumn(
        "source_hash",
        F.sha2(F.to_json(F.col("resource")), 256)
    )
    .select("source_hash")
    .first()["source_hash"]
)

DeltaTable.forName(spark, table_name).update(
    condition=(
        (F.col("resource_id") == test_id) &
        (F.col("is_current") == True) &
        (F.col("record_hash") != F.lit(source_hash))
    ),
    set={
        "is_current": "false",
        "valid_to": "current_timestamp()"
    }
)

print("Old version closure step completed.")

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 70, Finished, Available, Finished, False)

Old version closure step completed.


In [69]:
from pyspark.sql import functions as F

test_id = "018f428e-34f6-4707-8009-5ad742f901e7"

display(
    spark.table("Test_Patient_History_Clean")
    .filter(F.col("resource_id") == test_id)
    .select(
        "resource_id",
        F.col("resource.gender").alias("gender"),
        "is_current",
        "valid_from",
        "valid_to"
    )
    .orderBy("valid_from")
)

print(
    "Current versions:",
    spark.table("Test_Patient_History_Clean")
    .filter(
        (F.col("resource_id") == test_id) &
        (F.col("is_current") == True)
    )
    .count()
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 71, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, dddeb2ca-8eee-438b-a3ad-598a2de7b1e0)

Current versions: 0


In [70]:
from pyspark.sql import functions as F

table_name = "Test_Patient_History_Clean"
test_id = "018f428e-34f6-4707-8009-5ad742f901e7"

target = spark.table(table_name)
resource_schema = target.schema["resource"].dataType

source_record = (
    spark.table("Silver_Patient")
    .filter(F.col("resource_id") == test_id)
    .withColumn(
        "record_hash",
        F.sha2(F.to_json(F.col("resource")), 256)
    )
)

# Insert only if this patient has no current version
current_count = (
    target.filter(
        (F.col("resource_id") == test_id) &
        (F.col("is_current") == True)
    ).count()
)

if current_count == 0:
    (
        source_record
        .withColumn("is_current", F.lit(True))
        .withColumn("valid_from", F.current_timestamp())
        .withColumn("valid_to", F.lit(None).cast("timestamp"))
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(table_name)
    )
    print("Current source version inserted.")
else:
    print("A current version already exists; no insert performed.")

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 72, Finished, Available, Finished, False)

Current source version inserted.


In [71]:
display(
    spark.table("Test_Patient_History_Clean")
    .filter(F.col("resource_id") == test_id)
    .select(
        "resource_id",
        F.col("resource.gender").alias("gender"),
        "is_current",
        "valid_from",
        "valid_to"
    )
    .orderBy("valid_from")
)

print(
    "Current versions:",
    spark.table("Test_Patient_History_Clean")
    .filter(
        (F.col("resource_id") == test_id) &
        (F.col("is_current") == True)
    ).count()
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 73, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4017de49-d396-4fb9-bc21-eb2418bb8db9)

Current versions: 1


In [72]:
from pyspark.sql import functions as F

table_name = "Test_Patient_History_Clean"
test_id = "018f428e-34f6-4707-8009-5ad742f901e7"

history = spark.table(table_name)

patient_history = history.filter(
    F.col("resource_id") == test_id
)

print("Total versions:", patient_history.count())
print(
    "Current versions:",
    patient_history.filter(F.col("is_current") == True).count()
)
print(
    "Historical versions:",
    patient_history.filter(F.col("is_current") == False).count()
)

print(
    "Duplicate current IDs across table:",
    history.filter(F.col("is_current") == True)
    .groupBy("resource_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

display(
    patient_history.select(
        "resource_id",
        F.col("resource.gender").alias("gender"),
        "is_current",
        "valid_from",
        "valid_to"
    ).orderBy("valid_from")
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 74, Finished, Available, Finished, False)

Total versions: 3
Current versions: 1
Historical versions: 2
Duplicate current IDs across table: 0


SynapseWidget(Synapse.DataFrame, 11891390-985f-41db-9a02-4f44a4895272)

In [73]:
from pyspark.sql import functions as F

test_id = "018f428e-34f6-4707-8009-5ad742f901e7"

source = (
    spark.table("Silver_Patient")
    .filter(F.col("resource_id") == test_id)
    .withColumn(
        "source_hash",
        F.sha2(F.to_json(F.col("resource")), 256)
    )
    .select("resource_id", "source_hash")
)

current = (
    spark.table("Test_Patient_History_Clean")
    .filter(
        (F.col("resource_id") == test_id) &
        (F.col("is_current") == True)
    )
    .select(
        "resource_id",
        F.col("record_hash").alias("current_hash")
    )
)

display(
    source.join(current, "resource_id")
    .withColumn(
        "status",
        F.when(
            F.col("source_hash") == F.col("current_hash"),
            "UNCHANGED"
        ).otherwise("CHANGED")
    )
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 75, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a21bda42-1038-4073-b066-33bd7ee152b3)

In [74]:
from pyspark.sql import functions as F

table_name = "Test_Patient_History_Clean"
test_id = "018f428e-34f6-4707-8009-5ad742f901e7"

display(
    spark.table(table_name)
    .filter(F.col("resource_id") == test_id)
    .groupBy("is_current")
    .count()
)

print("Total versions:", 
      spark.table(table_name)
      .filter(F.col("resource_id") == test_id)
      .count())

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 76, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, be4715a8-ceef-4cb6-b9e1-06c38a81fc01)

Total versions: 3


In [75]:
from pyspark.sql import functions as F

table_name = "Test_Patient_History_Clean"
test_id = "018f428e-34f6-4707-8009-5ad742f901e7"

before_count = (
    spark.table(table_name)
    .filter(F.col("resource_id") == test_id)
    .count()
)

source_hash = (
    spark.table("Silver_Patient")
    .filter(F.col("resource_id") == test_id)
    .withColumn(
        "source_hash",
        F.sha2(F.to_json(F.col("resource")), 256)
    )
    .select("source_hash")
    .first()["source_hash"]
)

current_hash = (
    spark.table(table_name)
    .filter(
        (F.col("resource_id") == test_id) &
        (F.col("is_current") == True)
    )
    .select("record_hash")
    .first()["record_hash"]
)

if source_hash == current_hash:
    print("UNCHANGED - No new version required")
else:
    print("CHANGED - Version update required")

after_count = (
    spark.table(table_name)
    .filter(F.col("resource_id") == test_id)
    .count()
)

print("Versions before:", before_count)
print("Versions after:", after_count)
print("Idempotency check:", 
      "PASS" if before_count == after_count else "FAIL")

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 77, Finished, Available, Finished, False)

UNCHANGED - No new version required
Versions before: 3
Versions after: 3
Idempotency check: PASS


In [76]:
from pyspark.sql import functions as F

test_id = "018f428e-34f6-4707-8009-5ad742f901e7"

display(
    spark.table("Test_Patient_History_Clean")
    .filter(F.col("resource_id") == test_id)
    .select(
        "resource_id",
        F.col("resource.gender").alias("gender"),
        "is_current",
        "valid_from",
        "valid_to",
        "record_hash"
    )
    .orderBy("valid_from")
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 78, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 37340467-d487-4755-9665-e7606cb07199)

In [77]:
from delta.tables import DeltaTable

source_table = "Gold_Patient_History"
test_table = "Test_Patient_History_SCD2"

# Clean test table ko production history ki copy se initialize karein
spark.table(source_table) \
    .write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(test_table)

print("Test table created:", test_table)
print("Total records:", spark.table(test_table).count())
print(
    "Current records:",
    spark.table(test_table)
    .filter("is_current = true")
    .count()
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 79, Finished, Available, Finished, False)

Test table created: Test_Patient_History_SCD2
Total records: 9031
Current records: 9031


In [78]:

from delta.tables import DeltaTable
from pyspark.sql import functions as F

def apply_scd2(source_table, target_table):
    # Source: latest cleaned patient records
    source = (
        spark.table(source_table)
        .filter(F.col("resource_id").isNotNull())
        .dropDuplicates(["resource_id"])
        .withColumn(
            "record_hash",
            F.sha2(F.to_json(F.col("resource")), 256)
        )
    )

    # Current target records
    target = (
        spark.table(target_table)
        .filter(F.col("is_current") == True)
        .select(
            "resource_id",
            F.col("record_hash").alias("target_hash")
        )
    )

    # Identify new and changed records
    comparison = source.alias("s").join(
        target.alias("t"),
        on="resource_id",
        how="left"
    )

    changed = comparison.filter(
        F.col("target_hash").isNotNull() &
        (F.col("record_hash") != F.col("target_hash"))
    ).select("resource_id")

    new_records = comparison.filter(
        F.col("target_hash").isNull()
    ).select("resource_id")

    changed_count = changed.count()
    new_count = new_records.count()

    print("Changed records:", changed_count)
    print("New records:", new_count)
    print("Function defined successfully.")


StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 80, Finished, Available, Finished, False)

In [79]:

from delta.tables import DeltaTable
from pyspark.sql import functions as F

def apply_scd2(source_table, target_table):
    source = (
        spark.table(source_table)
        .filter(F.col("resource_id").isNotNull())
        .dropDuplicates(["resource_id"])
        .withColumn(
            "record_hash",
            F.sha2(F.to_json(F.col("resource")), 256)
        )
    )

    target_delta = DeltaTable.forName(spark, target_table)

    # Current target records ke saath source compare karein
    current = (
        spark.table(target_table)
        .filter(F.col("is_current") == True)
        .select(
            "resource_id",
            F.col("record_hash").alias("target_hash")
        )
    )

    comparison = source.join(current, "resource_id", "left")

    changed_ids = (
        comparison
        .filter(
            F.col("target_hash").isNotNull() &
            (F.col("record_hash") != F.col("target_hash"))
        )
        .select("resource_id")
        .distinct()
    )

    new_ids = (
        comparison
        .filter(F.col("target_hash").isNull())
        .select("resource_id")
        .distinct()
    )

    changed_count = changed_ids.count()
    new_count = new_ids.count()

    # Close previous current version for changed records
    if changed_count > 0:
        (
            target_delta.alias("t")
            .merge(
                changed_ids.alias("c"),
                "t.resource_id = c.resource_id AND t.is_current = true"
            )
            .whenMatchedUpdate(set={
                "is_current": "false",
                "valid_to": "current_timestamp()"
            })
            .execute()
        )

    # New and changed source rows ko current versions ke roop mein insert karein
    rows_to_insert = (
        source.join(
            changed_ids.union(new_ids).distinct(),
            "resource_id",
            "inner"
        )
        .withColumn("is_current", F.lit(True))
        .withColumn("valid_from", F.current_timestamp())
        .withColumn("valid_to", F.lit(None).cast("timestamp"))
    )

    insert_count = rows_to_insert.count()

    if insert_count > 0:
        rows_to_insert.write.format("delta") \
            .mode("append") \
            .saveAsTable(target_table)

    print("Changed records:", changed_count)
    print("New records:", new_count)
    print("New current versions inserted:", insert_count)
    print("SCD Type 2 processing completed.")


StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 81, Finished, Available, Finished, False)

In [80]:
from pyspark.sql import functions as F

source_table = "Silver_Patient"
target_table = "Test_Patient_History_SCD2"

source = (
    spark.table(source_table)
    .filter(F.col("resource_id").isNotNull())
    .withColumn(
        "record_hash",
        F.sha2(F.to_json(F.col("resource")), 256)
    )
    .select("resource_id", "record_hash")
    .dropDuplicates(["resource_id"])
)

current = (
    spark.table(target_table)
    .filter(F.col("is_current") == True)
    .select(
        "resource_id",
        F.col("record_hash").alias("target_hash")
    )
)

comparison = source.join(current, "resource_id", "left")

print(
    "Changed records:",
    comparison.filter(
        F.col("target_hash").isNotNull() &
        (F.col("record_hash") != F.col("target_hash"))
    ).count()
)

print(
    "New records:",
    comparison.filter(F.col("target_hash").isNull()).count()
)

print("Source records:", source.count())
print("Current target records:", current.count())

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 82, Finished, Available, Finished, False)

Changed records: 0
New records: 0
Source records: 9031
Current target records: 9031


In [81]:
from pyspark.sql import functions as F

test_id = (
    spark.table("Silver_Patient")
    .select("resource_id")
    .where(F.col("resource_id").isNotNull())
    .first()["resource_id"]
)

print("Test patient ID:", test_id)

display(
    spark.table("Test_Patient_History_SCD2")
    .filter(
        (F.col("resource_id") == test_id) &
        (F.col("is_current") == True)
    )
    .select(
        "resource_id",
        F.col("resource.gender").alias("gender"),
        "is_current",
        "valid_from",
        "valid_to",
        "record_hash"
    )
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 83, Finished, Available, Finished, False)

Test patient ID: 045ed820-8c47-477d-a6d9-25b69e411b7d


SynapseWidget(Synapse.DataFrame, 50db8f1a-bebf-4b04-9210-e73a47ddac20)

In [82]:
from pyspark.sql import functions as F

test_id = "045ed820-8c47-477d-a6d9-25b69e411b7d"

source = (
    spark.table("Silver_Patient")
    .filter(F.col("resource_id") == test_id)
    .withColumn(
        "source_hash",
        F.sha2(F.to_json(F.col("resource")), 256)
    )
    .select("resource_id", "source_hash")
)

target = (
    spark.table("Test_Patient_History_SCD2")
    .filter(
        (F.col("resource_id") == test_id) &
        (F.col("is_current") == True)
    )
    .select(
        "resource_id",
        F.col("record_hash").alias("target_hash"),
        F.col("resource.gender").alias("target_gender")
    )
)

display(source.join(target, "resource_id"))

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 84, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 72b877a7-b284-47f0-987a-38862598a654)

In [83]:
from pyspark.sql import functions as F

test_id = "045ed820-8c47-477d-a6d9-25b69e411b7d"

source = (
    spark.table("Silver_Patient")
    .filter(F.col("resource_id") == test_id)
)

# Sirf test ke liye resource JSON mein gender male -> female
changed_source = (
    source
    .withColumn(
        "_resource_json",
        F.regexp_replace(
            F.to_json(F.col("resource")),
            '"gender":"male"',
            '"gender":"female"'
        )
    )
    .withColumn(
        "resource",
        F.from_json(
            F.col("_resource_json"),
            source.schema["resource"].dataType
        )
    )
    .drop("_resource_json")
    .withColumn(
        "record_hash",
        F.sha2(F.to_json(F.col("resource")), 256)
    )
)

display(
    changed_source.select(
        "resource_id",
        F.col("resource.gender").alias("simulated_gender"),
        "record_hash"
    )
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 85, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2c8627ca-7008-4ddd-9c48-2e8b27b9fc91)

In [84]:
from pyspark.sql import functions as F

test_id = "045ed820-8c47-477d-a6d9-25b69e411b7d"

display(
    spark.table("Test_Patient_History_SCD2")
    .filter(F.col("resource_id") == test_id)
    .select(
        "resource_id",
        F.col("resource.gender").alias("gender"),
        "is_current",
        "valid_from",
        "valid_to",
        "record_hash"
    )
    .orderBy("valid_from")
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 86, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ad0a002a-482c-48ee-b84d-afa6923da306)

In [85]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

def apply_scd2_df(source_df, target_table):
    source = (
        source_df
        .filter(F.col("resource_id").isNotNull())
        .dropDuplicates(["resource_id"])
        .withColumn(
            "record_hash",
            F.sha2(F.to_json(F.col("resource")), 256)
        )
    )

    current = (
        spark.table(target_table)
        .filter(F.col("is_current") == True)
        .select(
            "resource_id",
            F.col("record_hash").alias("target_hash")
        )
    )

    comparison = source.join(current, "resource_id", "left")

    changed_ids = (
        comparison
        .filter(
            F.col("target_hash").isNotNull() &
            (F.col("record_hash") != F.col("target_hash"))
        )
        .select("resource_id")
        .distinct()
    )

    new_ids = (
        comparison
        .filter(F.col("target_hash").isNull())
        .select("resource_id")
        .distinct()
    )

    changed_count = changed_ids.count()
    new_count = new_ids.count()

    print("Changed:", changed_count)
    print("New:", new_count)

    if changed_count > 0:
        DeltaTable.forName(spark, target_table).alias("t").merge(
            changed_ids.alias("c"),
            "t.resource_id = c.resource_id AND t.is_current = true"
        ).whenMatchedUpdate(set={
            "is_current": "false",
            "valid_to": "current_timestamp()"
        }).execute()

    ids_to_insert = changed_ids.union(new_ids).distinct()

    rows_to_insert = (
        source.join(ids_to_insert, "resource_id", "inner")
        .withColumn("is_current", F.lit(True))
        .withColumn("valid_from", F.current_timestamp())
        .withColumn("valid_to", F.lit(None).cast("timestamp"))
    )

    insert_count = rows_to_insert.count()

    if insert_count > 0:
        rows_to_insert.write.format("delta") \
            .mode("append").saveAsTable(target_table)

    print("Inserted versions:", insert_count)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 87, Finished, Available, Finished, False)

In [86]:
test_id = "045ed820-8c47-477d-a6d9-25b69e411b7d"
target_table = "Test_Patient_History_SCD2"

# Sirf selected patient ka synthetic change pass kar rahe hain
test_source = changed_source.filter(
    F.col("resource_id") == test_id
)

apply_scd2_df(test_source, target_table)

display(
    spark.table(target_table)
    .filter(F.col("resource_id") == test_id)
    .select(
        "resource_id",
        F.col("resource.gender").alias("gender"),
        "is_current",
        "valid_from",
        "valid_to",
        "record_hash"
    )
    .orderBy("valid_from")
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 88, Finished, Available, Finished, False)

Changed: 1
New: 0
Inserted versions: 1


SynapseWidget(Synapse.DataFrame, 15fe5d66-0f07-4fe8-af90-94f6ff267691)

In [87]:
test_id = "045ed820-8c47-477d-a6d9-25b69e411b7d"
target_table = "Test_Patient_History_SCD2"

before_count = (
    spark.table(target_table)
    .filter(F.col("resource_id") == test_id)
    .count()
)

apply_scd2_df(
    changed_source.filter(F.col("resource_id") == test_id),
    target_table
)

after_count = (
    spark.table(target_table)
    .filter(F.col("resource_id") == test_id)
    .count()
)

print("Versions before:", before_count)
print("Versions after:", after_count)
print(
    "Idempotency:",
    "PASS" if before_count == after_count else "FAIL"
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 89, Finished, Available, Finished, False)

Changed: 0
New: 0
Inserted versions: 0
Versions before: 2
Versions after: 2
Idempotency: PASS


In [88]:
from pyspark.sql import functions as F

test_id = "045ed820-8c47-477d-a6d9-25b69e411b7d"
table_name = "Test_Patient_History_SCD2"

patient_history = (
    spark.table(table_name)
    .filter(F.col("resource_id") == test_id)
)

total_versions = patient_history.count()

current_count = patient_history.filter(
    F.col("is_current") == True
).count()

historical_count = patient_history.filter(
    F.col("is_current") == False
).count()

invalid_current_dates = patient_history.filter(
    (F.col("is_current") == True) &
    F.col("valid_to").isNotNull()
).count()

print("Total versions:", total_versions)
print("Current versions:", current_count)
print("Historical versions:", historical_count)
print("Current records with non-null valid_to:", invalid_current_dates)

print(
    "SCD2 validation:",
    "PASS" if (
        current_count == 1 and
        total_versions == current_count + historical_count and
        invalid_current_dates == 0
    ) else "FAIL"
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 90, Finished, Available, Finished, False)

Total versions: 2
Current versions: 1
Historical versions: 1
Current records with non-null valid_to: 0
SCD2 validation: PASS


In [89]:
from notebookutils import mssparkutils

base_path = "Files/raw"

for resource in ["Patient", "Encounter", "Observation", "Condition"]:
    resource_path = f"{base_path}/{resource}"

    try:
        items = mssparkutils.fs.ls(resource_path)
        print(f"{resource}: folder exists, items = {len(items)}")
    except Exception as e:
        print(f"{resource}: folder missing or unreadable")

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 91, Finished, Available, Finished, False)

Patient: folder exists, items = 1
Encounter: folder exists, items = 1
Observation: folder exists, items = 1
Condition: folder exists, items = 1


In [90]:
from notebookutils import mssparkutils

path = "Files/raw/Patient"

for item in mssparkutils.fs.ls(path):
    print(item.name, "| Directory:", item.isDir, "| Path:", item.path)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 92, Finished, Available, Finished, False)

load_date=2026-10-25 | Directory: True | Path: abfss://d621f477-fcf6-44f4-b9da-2e279f03b748@onelake.dfs.fabric.microsoft.com/7c5272cd-1a9e-4e1a-88c5-395a8938a85a/Files/raw/Patient/load_date=2026-10-25


In [91]:
from notebookutils import mssparkutils

path = "Files/raw/Patient/load_date=2026-10-25"

for item in mssparkutils.fs.ls(path):
    print(
        item.name,
        "| Directory:", item.isDir,
        "| Path:", item.path
    )

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 93, Finished, Available, Finished, False)

run_id=2243bed4-2e0f-470a-be84-00d883f472ab | Directory: True | Path: abfss://d621f477-fcf6-44f4-b9da-2e279f03b748@onelake.dfs.fabric.microsoft.com/7c5272cd-1a9e-4e1a-88c5-395a8938a85a/Files/raw/Patient/load_date=2026-10-25/run_id=2243bed4-2e0f-470a-be84-00d883f472ab
run_id=d227640b-3866-48a5-9e2b-8cb19a448438 | Directory: True | Path: abfss://d621f477-fcf6-44f4-b9da-2e279f03b748@onelake.dfs.fabric.microsoft.com/7c5272cd-1a9e-4e1a-88c5-395a8938a85a/Files/raw/Patient/load_date=2026-10-25/run_id=d227640b-3866-48a5-9e2b-8cb19a448438


In [92]:
from notebookutils import mssparkutils

base_path = "Files/raw/Patient/load_date=2026-10-25"

for run in mssparkutils.fs.ls(base_path):
    if run.isDir:
        items = mssparkutils.fs.ls(run.path)
        json_files = [
            item for item in items
            if not item.isDir and item.name.endswith(".json")
        ]

        print("Run:", run.name)
        print("Total items:", len(items))
        print("JSON files:", len(json_files))
        print("---")

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 94, Finished, Available, Finished, False)

Run: run_id=2243bed4-2e0f-470a-be84-00d883f472ab
Total items: 91
JSON files: 91
---
Run: run_id=d227640b-3866-48a5-9e2b-8cb19a448438
Total items: 1
JSON files: 1
---


In [93]:
import requests
from datetime import datetime, timedelta, timezone

base_url = "https://hapi.fhir.org/baseR4"
resource_type = "Patient"

# Recent 2 days ka UTC cutoff
cutoff = (
    datetime.now(timezone.utc) - timedelta(days=2)
).strftime("%Y-%m-%dT%H:%M:%SZ")

url = f"{base_url}/{resource_type}"

params = {
    "_lastUpdated": f"ge{cutoff}",
    "_count": 10,
    "_format": "json"
}

response = requests.get(url, params=params, timeout=60)

print("HTTP Status:", response.status_code)
print("Request URL:", response.url)

if response.status_code == 200:
    bundle = response.json()
    entries = bundle.get("entry", [])

    print("Bundle type:", bundle.get("type"))
    print("Total reported by API:", bundle.get("total"))
    print("Records in first page:", len(entries))
    print("Cutoff UTC:", cutoff)
else:
    print("Response:", response.text[:1000])

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 95, Finished, Available, Finished, False)

HTTP Status: 200
Request URL: https://hapi.fhir.org/baseR4/Patient?_lastUpdated=ge2026-10-23T19%3A12%3A43Z&_count=10&_format=json
Bundle type: searchset
Total reported by API: None
Records in first page: 10
Cutoff UTC: 2026-10-23T19:12:43Z


In [94]:
# Previous cell ke bundle variable ka use karein

entries = bundle.get("entry", [])

print("First page records:", len(entries))

if entries:
    for i, entry in enumerate(entries[:5], start=1):
        resource = entry.get("resource", {})
        print(
            i,
            "| ID:", resource.get("id"),
            "| Last updated:", resource.get("meta", {}).get("lastUpdated")
        )

next_links = [
    link.get("url")
    for link in bundle.get("link", [])
    if link.get("relation") == "next"
]

print("Next page available:", bool(next_links))
if next_links:
    print("Next URL:", next_links[0])

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 96, Finished, Available, Finished, False)

First page records: 10
1 | ID: 259771 | Last updated: 2026-10-23T15:16:44.044-04:00
2 | ID: 260167 | Last updated: 2026-10-23T15:20:28.424-04:00
3 | ID: 260362 | Last updated: 2026-10-23T15:25:02.216-04:00
4 | ID: 260367 | Last updated: 2026-10-23T15:25:02.216-04:00
5 | ID: 260405 | Last updated: 2026-10-23T15:25:02.216-04:00
Next page available: True
Next URL: https://hapi.fhir.org/baseR4?_getpages=06016d59-7ae9-4eb1-8b87-761199a61abb&_getpagesoffset=10&_count=10&_format=json&_pretty=true&_bundletype=searchset


In [95]:

import requests
import json
import uuid
from datetime import datetime, timedelta, timezone
from notebookutils import mssparkutils

FHIR_BASE = "https://hapi.fhir.org/baseR4"
RAW_BASE = "Files/raw"

def ingest_resource_incremental(resource_type, lookback_days=2, page_size=100):
    run_id = str(uuid.uuid4())
    load_date = datetime.now(timezone.utc).strftime("%Y-%m-%d")
    cutoff = (
        datetime.now(timezone.utc) - timedelta(days=lookback_days)
    ).strftime("%Y-%m-%dT%H:%M:%SZ")

    output_dir = (
        f"{RAW_BASE}/{resource_type}/"
        f"load_date={load_date}/run_id={run_id}"
    )
    mssparkutils.fs.mkdirs(output_dir)

    url = f"{FHIR_BASE}/{resource_type}"
    params = {
        "_lastUpdated": f"ge{cutoff}",
        "_count": page_size,
        "_format": "json"
    }

    session = requests.Session()
    page_num = 0
    total_saved = 0

    while url:
        response = session.get(
            url,
            params=params if page_num == 0 else None,
            timeout=90
        )
        response.raise_for_status()
        bundle = response.json()

        # Store each page's original Bundle JSON
        page_num += 1
        file_path = f"{output_dir}/page_{page_num}.json"

        mssparkutils.fs.put(
            file_path,
            json.dumps(bundle, ensure_ascii=False),
            True
        )

        entries = bundle.get("entry", [])
        total_saved += len(entries)

        next_url = next(
            (
                link["url"]
                for link in bundle.get("link", [])
                if link.get("relation") == "next"
            ),
            None
        )

        url = next_url

    print("Resource:", resource_type)
    print("Cutoff UTC:", cutoff)
    print("Run ID:", run_id)
    print("Pages saved:", page_num)
    print("Bundle entries saved:", total_saved)
    print("Output folder:", output_dir)


StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 97, Finished, Available, Finished, False)

In [96]:
ingest_resource_incremental(
    resource_type="Patient",
    lookback_days=2,
    page_size=100
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 98, Finished, Available, Finished, False)

Resource: Patient
Cutoff UTC: 2026-10-23T19:15:13Z
Run ID: cf3a17d3-c298-4c0b-80d3-8e17f392d522
Pages saved: 7
Bundle entries saved: 638
Output folder: Files/raw/Patient/load_date=2026-10-25/run_id=cf3a17d3-c298-4c0b-80d3-8e17f392d522


In [97]:
from notebookutils import mssparkutils

output_dir = (
    "Files/raw/Patient/load_date=2026-10-25/"
    "run_id=cf3a17d3-c298-4c0b-80d3-8e17f392d522"
)

files = mssparkutils.fs.ls(output_dir)

json_files = [
    f for f in files
    if not f.isDir and f.name.endswith(".json")
]

print("JSON files found:", len(json_files))

for f in json_files:
    print(f.name, "| Size:", f.size)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 99, Finished, Available, Finished, False)

JSON files found: 7
page_1.json | Size: 84632
page_2.json | Size: 99335
page_3.json | Size: 71066
page_4.json | Size: 70727
page_5.json | Size: 74857
page_6.json | Size: 76662
page_7.json | Size: 29028


In [98]:
ingest_resource_incremental(
    resource_type="Encounter",
    lookback_days=2,
    page_size=100
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 100, Finished, Available, Finished, False)

Resource: Encounter
Cutoff UTC: 2026-10-23T19:16:26Z
Run ID: 44bc2f1e-9e6b-4a4c-8910-ed871e160a54
Pages saved: 3
Bundle entries saved: 239
Output folder: Files/raw/Encounter/load_date=2026-10-25/run_id=44bc2f1e-9e6b-4a4c-8910-ed871e160a54


In [99]:
ingest_resource_incremental(
    resource_type="Observation",
    lookback_days=2,
    page_size=100
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 101, Finished, Available, Finished, False)

Resource: Observation
Cutoff UTC: 2026-10-23T19:17:00Z
Run ID: c25bd56c-a9ed-4527-bc38-a7cf7334044d
Pages saved: 31
Bundle entries saved: 3058
Output folder: Files/raw/Observation/load_date=2026-10-25/run_id=c25bd56c-a9ed-4527-bc38-a7cf7334044d


In [100]:
ingest_resource_incremental(
    resource_type="Condition",
    lookback_days=2,
    page_size=100
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 102, Finished, Available, Finished, False)

Resource: Condition
Cutoff UTC: 2026-10-23T19:17:49Z
Run ID: 1260e542-3b11-4445-8c4e-c0ab51f8aa90
Pages saved: 2
Bundle entries saved: 120
Output folder: Files/raw/Condition/load_date=2026-10-25/run_id=1260e542-3b11-4445-8c4e-c0ab51f8aa90


In [101]:
from pyspark.sql.types import (
    StructType, StructField, StringType,
    IntegerType, TimestampType
)

metadata_schema = StructType([
    StructField("resource_type", StringType(), True),
    StructField("run_id", StringType(), True),
    StructField("cutoff_utc", StringType(), True),
    StructField("api_url_or_params", StringType(), True),
    StructField("pages_saved", IntegerType(), True),
    StructField("records_fetched", IntegerType(), True),
    StructField("extraction_timestamp", TimestampType(), True),
    StructField("save_timestamp", TimestampType(), True),
    StructField("status", StringType(), True)
])

metadata_df = spark.createDataFrame([], metadata_schema)

print("Metadata schema created successfully")
metadata_df.printSchema()

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 103, Finished, Available, Finished, False)

Metadata schema created successfully
root
 |-- resource_type: string (nullable = true)
 |-- run_id: string (nullable = true)
 |-- cutoff_utc: string (nullable = true)
 |-- api_url_or_params: string (nullable = true)
 |-- pages_saved: integer (nullable = true)
 |-- records_fetched: integer (nullable = true)
 |-- extraction_timestamp: timestamp (nullable = true)
 |-- save_timestamp: timestamp (nullable = true)
 |-- status: string (nullable = true)



In [102]:

from pyspark.sql import Row

metadata_records = [
    Row(
        resource_type="Patient",
        run_id="cf3a17d3-c298-4c0b-80d3-8e17f392d522",
        cutoff_utc="2026-10-23T19:15:13Z",
        api_url_or_params="FHIR Patient; _lastUpdated=ge2026-10-23T19:15:13Z; _count=100; _format=json",
        pages_saved=7,
        records_fetched=638,
        extraction_timestamp=None,
        save_timestamp=None,
        status="SUCCESS"
    ),
    Row(
        resource_type="Encounter",
        run_id="44bc2f1e-9e6b-4a4c-8910-ed871e160a54",
        cutoff_utc="2026-10-23T19:16:26Z",
        api_url_or_params="FHIR Encounter; _lastUpdated=ge2026-10-23T19:16:26Z; _count=100; _format=json",
        pages_saved=3,
        records_fetched=239,
        extraction_timestamp=None,
        save_timestamp=None,
        status="SUCCESS"
    ),
    Row(
        resource_type="Observation",
        run_id="c25bd56c-a9ed-4527-bc38-a7cf7334044d",
        cutoff_utc="2026-10-23T19:17:00Z",
        api_url_or_params="FHIR Observation; _lastUpdated=ge2026-10-23T19:17:00Z; _count=100; _format=json",
        pages_saved=31,
        records_fetched=3058,
        extraction_timestamp=None,
        save_timestamp=None,
        status="SUCCESS"
    ),
    Row(
        resource_type="Condition",
        run_id="1260e542-3b11-4445-8c4e-c0ab51f8aa90",
        cutoff_utc="2026-10-23T19:17:49Z",
        api_url_or_params="FHIR Condition; _lastUpdated=ge2026-10-23T19:17:49Z; _count=100; _format=json",
        pages_saved=2,
        records_fetched=120,
        extraction_timestamp=None,
        save_timestamp=None,
        status="SUCCESS"
    )
]

metadata_df = spark.createDataFrame(metadata_records, schema=metadata_schema)

display(
    metadata_df.select(
        "resource_type",
        "pages_saved",
        "records_fetched",
        "status"
    )
)

print("Total pages saved:", metadata_df.agg({"pages_saved": "sum"}).first()[0])
print("Total records fetched:", metadata_df.agg({"records_fetched": "sum"}).first()[0])


StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 104, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7f1ffc1b-6399-498a-9a68-0b794478ca94)

Total pages saved: 43
Total records fetched: 4055


In [103]:

metadata_table = "Ingestion_Run_Metadata"

if spark.catalog.tableExists(metadata_table):
    print("Metadata table already exists. Appending new run records.")
else:
    print("Metadata table does not exist. Creating it with the current records.")

metadata_df.write.format("delta").mode("append").saveAsTable(metadata_table)

print("Metadata saved successfully.")

display(
    spark.table(metadata_table)
    .select(
        "resource_type",
        "run_id",
        "pages_saved",
        "records_fetched",
        "status"
    )
    .orderBy("resource_type")
)


StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 105, Finished, Available, Finished, False)

Metadata table does not exist. Creating it with the current records.
Metadata saved successfully.


SynapseWidget(Synapse.DataFrame, 4cbb12e5-069a-4f26-ae23-bf8955e70f73)

In [104]:

from pyspark.sql import functions as F

metadata_check = spark.table("Ingestion_Run_Metadata")

metadata_check.select(
    F.count("*").alias("total_runs"),
    F.countDistinct("run_id").alias("unique_runs"),
    F.sum("pages_saved").alias("total_pages"),
    F.sum("records_fetched").alias("total_records_fetched"),
    F.sum(
        F.when(F.col("status") == "SUCCESS", 1).otherwise(0)
    ).alias("successful_runs")
).show(truncate=False)

duplicate_runs = (
    metadata_check.groupBy("run_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate run IDs:")
duplicate_runs.show(truncate=False)


StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 106, Finished, Available, Finished, False)

+----------+-----------+-----------+---------------------+---------------+
|total_runs|unique_runs|total_pages|total_records_fetched|successful_runs|
+----------+-----------+-----------+---------------------+---------------+
|4         |4          |43         |4055                 |4              |
+----------+-----------+-----------+---------------------+---------------+

Duplicate run IDs:
+------+-----+
|run_id|count|
+------+-----+
+------+-----+



In [105]:

print("Incremental ingestion function available:",
      callable(globals().get("ingest_resource_incremental")))

print("Metadata table available:",
      spark.catalog.tableExists("Ingestion_Run_Metadata"))



StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 107, Finished, Available, Finished, False)

Incremental ingestion function available: True
Metadata table available: True


In [106]:

import inspect

print(inspect.getsource(ingest_resource_incremental))


StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 108, Finished, Available, Finished, False)

def ingest_resource_incremental(resource_type, lookback_days=2, page_size=100):
    run_id = str(uuid.uuid4())
    load_date = datetime.now(timezone.utc).strftime("%Y-%m-%d")
    cutoff = (
        datetime.now(timezone.utc) - timedelta(days=lookback_days)
    ).strftime("%Y-%m-%dT%H:%M:%SZ")

    output_dir = (
        f"{RAW_BASE}/{resource_type}/"
        f"load_date={load_date}/run_id={run_id}"
    )
    mssparkutils.fs.mkdirs(output_dir)

    url = f"{FHIR_BASE}/{resource_type}"
    params = {
        "_lastUpdated": f"ge{cutoff}",
        "_count": page_size,
        "_format": "json"
    }

    session = requests.Session()
    page_num = 0
    total_saved = 0

    while url:
        response = session.get(
            url,
            params=params if page_num == 0 else None,
            timeout=90
        )
        response.raise_for_status()
        bundle = response.json()

        # Store each page's original Bundle JSON
        page_num += 1
        file_path = f"{output_di

In [107]:

import uuid
import json
import requests
from datetime import datetime, timezone, timedelta
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField, StringType,
    IntegerType, TimestampType
)

def ingest_resource_incremental(resource_type, lookback_days=2, page_size=100):
    run_id = str(uuid.uuid4())
    started_at = datetime.now(timezone.utc)
    load_date = started_at.strftime("%Y-%m-%d")
    cutoff = (
        started_at - timedelta(days=lookback_days)
    ).strftime("%Y-%m-%dT%H:%M:%SZ")

    output_dir = (
        f"{RAW_BASE}/{resource_type}/"
        f"load_date={load_date}/run_id={run_id}"
    )

    url = f"{FHIR_BASE}/{resource_type}"
    params = {
        "_lastUpdated": f"ge{cutoff}",
        "_count": page_size,
        "_format": "json"
    }

    page_num = 0
    total_saved = 0
    status = "SUCCESS"
    error_message = None

    try:
        mssparkutils.fs.mkdirs(output_dir)
        session = requests.Session()

        while url:
            response = session.get(
                url,
                params=params if page_num == 0 else None,
                timeout=90
            )
            response.raise_for_status()
            bundle = response.json()

            next_page_num = page_num + 1
            file_path = f"{output_dir}/page_{next_page_num}.json"

            # Save original Bundle JSON
            mssparkutils.fs.put(
                file_path,
                json.dumps(bundle, ensure_ascii=False),
                True
            )

            page_num = next_page_num
            total_saved += len(bundle.get("entry", []))

            url = next(
                (
                    link["url"]
                    for link in bundle.get("link", [])
                    if link.get("relation") == "next"
                ),
                None
            )

            params = None

    except Exception as e:
        status = "FAILED"
        error_message = str(e)[:2000]

    finally:
        finished_at = datetime.now(timezone.utc)

        metadata_schema = StructType([
            StructField("resource_type", StringType(), True),
            StructField("run_id", StringType(), True),
            StructField("cutoff_utc", StringType(), True),
            StructField("api_url_or_params", StringType(), True),
            StructField("pages_saved", IntegerType(), True),
            StructField("records_fetched", IntegerType(), True),
            StructField("extraction_timestamp", TimestampType(), True),
            StructField("save_timestamp", TimestampType(), True),
            StructField("status", StringType(), True)
        ])

        metadata_row = [Row(
            resource_type=resource_type,
            run_id=run_id,
            cutoff_utc=cutoff,
            api_url_or_params=(
                f"{FHIR_BASE}/{resource_type}?_lastUpdated=ge{cutoff}"
                f"&_count={page_size}&_format=json"
            ),
            pages_saved=page_num,
            records_fetched=total_saved,
            extraction_timestamp=started_at.replace(tzinfo=None),
            save_timestamp=finished_at.replace(tzinfo=None),
            status=status
        )]

        run_metadata_df = spark.createDataFrame(
            metadata_row,
            schema=metadata_schema
        )

        run_metadata_df.write.format("delta").mode("append").saveAsTable(
            "Ingestion_Run_Metadata"
        )

    print("Resource:", resource_type)
    print("Run ID:", run_id)
    print("Cutoff UTC:", cutoff)
    print("Pages saved:", page_num)
    print("Bundle entries saved:", total_saved)
    print("Status:", status)
    print("Output folder:", output_dir)

    if error_message:
        print("Error:", error_message)
        raise RuntimeError(
            f"Ingestion failed for {resource_type}. Run ID: {run_id}"
        )

    return run_id


StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 109, Finished, Available, Finished, False)

In [108]:

print(
    "Updated function available:",
    callable(globals().get("ingest_resource_incremental"))
)

print(
    "Metadata table available:",
    spark.catalog.tableExists("Ingestion_Run_Metadata")
)

print("Existing metadata row count:",
      spark.table("Ingestion_Run_Metadata").count())


StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 110, Finished, Available, Finished, False)

Updated function available: True
Metadata table available: True
Existing metadata row count: 4


In [109]:

from datetime import datetime, timezone
from pyspark.sql import Row

test_run_id = "TEST-METADATA-" + str(uuid.uuid4())
test_time = datetime.now(timezone.utc).replace(tzinfo=None)

test_metadata = spark.createDataFrame(
    [Row(
        resource_type="TEST",
        run_id=test_run_id,
        cutoff_utc=None,
        api_url_or_params="Metadata logging validation only; no API call",
        pages_saved=0,
        records_fetched=0,
        extraction_timestamp=test_time,
        save_timestamp=test_time,
        status="TEST"
    )],
    schema=metadata_schema
)

test_metadata.write.format("delta").mode("append").saveAsTable(
    "Ingestion_Run_Metadata"
)

display(
    spark.table("Ingestion_Run_Metadata")
    .filter(F.col("run_id") == test_run_id)
    .select(
        "resource_type",
        "run_id",
        "extraction_timestamp",
        "save_timestamp",
        "status"
    )
)


StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 111, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a7f352bd-7794-468e-85a1-b93b75bb1a4c)

In [110]:

test_run_id = "TEST-METADATA-7c5dd1d8-3e0b-417e-95a0-1d5f3096f6fb"

from delta.tables import DeltaTable

metadata_delta = DeltaTable.forName(spark, "Ingestion_Run_Metadata")

metadata_delta.delete(
    F.col("run_id") == test_run_id
)

print("Remaining metadata rows:",
      spark.table("Ingestion_Run_Metadata").count())

display(
    spark.table("Ingestion_Run_Metadata")
    .groupBy("status")
    .count()
)


StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 112, Finished, Available, Finished, False)

Remaining metadata rows: 4


SynapseWidget(Synapse.DataFrame, c8e2d1f8-fd2b-4815-9d73-ddb7f01e8e07)

In [111]:

tables_to_check = [
    "Bronze_Patient",
    "Bronze_Encounter",
    "Bronze_Observation",
    "Bronze_Condition",
    "Silver_Patient",
    "Silver_Encounter",
    "Silver_Observation",
    "Silver_Condition",
    "Gold_Healthcare_Summary",
    "Gold_Patient_History",
    "Test_Patient_History_SCD2",
    "Ingestion_Run_Metadata"
]

for table_name in tables_to_check:
    exists = spark.catalog.tableExists(table_name)
    if exists:
        print(f"{table_name}: EXISTS | rows={spark.table(table_name).count()}")
    else:
        print(f"{table_name}: NOT FOUND")


StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 113, Finished, Available, Finished, False)

Bronze_Patient: EXISTS | rows=9041
Bronze_Encounter: EXISTS | rows=10603
Bronze_Observation: EXISTS | rows=58122
Bronze_Condition: EXISTS | rows=5944
Silver_Patient: EXISTS | rows=9031
Silver_Encounter: EXISTS | rows=10603
Silver_Observation: EXISTS | rows=58122
Silver_Condition: EXISTS | rows=5944
Gold_Healthcare_Summary: EXISTS | rows=4
Gold_Patient_History: EXISTS | rows=9031
Test_Patient_History_SCD2: EXISTS | rows=9032
Ingestion_Run_Metadata: EXISTS | rows=4


In [112]:

from pyspark.sql import functions as F

history_df = spark.table("Test_Patient_History_SCD2")

history_df.agg(
    F.count("*").alias("total_versions"),
    F.sum(
        F.when(F.col("is_current") == True, 1).otherwise(0)
    ).alias("current_versions"),
    F.sum(
        F.when(F.col("is_current") == False, 1).otherwise(0)
    ).alias("historical_versions"),
    F.sum(
        F.when(
            (F.col("is_current") == True) &
            F.col("valid_to").isNotNull(),
            1
        ).otherwise(0)
    ).alias("invalid_current_rows"),
    F.sum(
        F.when(
            (F.col("is_current") == False) &
            F.col("valid_to").isNull(),
            1
        ).otherwise(0)
    ).alias("invalid_historical_rows")
).show(truncate=False)

duplicate_current_ids = (
    history_df
    .filter(F.col("is_current") == True)
    .groupBy("resource_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Patient IDs with multiple current versions:")
duplicate_current_ids.show(truncate=False)


StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 114, Finished, Available, Finished, False)

+--------------+----------------+-------------------+--------------------+-----------------------+
|total_versions|current_versions|historical_versions|invalid_current_rows|invalid_historical_rows|
+--------------+----------------+-------------------+--------------------+-----------------------+
|9032          |9031            |1                  |0                   |0                      |
+--------------+----------------+-------------------+--------------------+-----------------------+

Patient IDs with multiple current versions:
+-----------+-----+
|resource_id|count|
+-----------+-----+
+-----------+-----+



In [113]:

incremental_runs = {
    "Patient": "Files/raw/Patient/load_date=2026-10-25/run_id=cf3a17d3-c298-4c0b-80d3-8e17f392d522",
    "Encounter": "Files/raw/Encounter/load_date=2026-10-25/run_id=44bc2f1e-9e6b-4a4c-8910-ed871e160a54",
    "Observation": "Files/raw/Observation/load_date=2026-10-25/run_id=c25bd56c-a9ed-4527-bc38-a7cf7334044d",
    "Condition": "Files/raw/Condition/load_date=2026-10-25/run_id=1260e542-3b11-4445-8c4e-c0ab51f8aa90"
}

for resource, folder in incremental_runs.items():
    files = [
        item for item in mssparkutils.fs.ls(folder)
        if item.name.endswith(".json")
    ]
    print(f"{resource}: {len(files)} JSON pages")


StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 115, Finished, Available, Finished, False)

Patient: 7 JSON pages
Encounter: 3 JSON pages
Observation: 31 JSON pages
Condition: 2 JSON pages


In [114]:

patient_page_path = (
    "Files/raw/Patient/load_date=2026-10-25/"
    "run_id=cf3a17d3-c298-4c0b-80d3-8e17f392d522/page_1.json"
)

patient_page_text = mssparkutils.fs.head(patient_page_path, 5000)

print(patient_page_text)


StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 116, Finished, Available, Finished, False)

{"resourceType": "Bundle", "id": "a2a7c4f0-4f28-4683-b03b-8a559f81929f", "meta": {"lastUpdated": "2026-10-25T15:15:14.350-04:00"}, "type": "searchset", "link": [{"relation": "self", "url": "https://hapi.fhir.org/baseR4/Patient?_count=100&_format=json&_lastUpdated=ge2026-10-23T19%3A15%3A13Z"}, {"relation": "next", "url": "https://hapi.fhir.org/baseR4?_getpages=a2a7c4f0-4f28-4683-b03b-8a559f81929f&_getpagesoffset=100&_count=100&_format=json&_pretty=true&_bundletype=searchset"}], "entry": [{"fullUrl": "https://hapi.fhir.org/baseR4/Patient/259771", "resource": {"resourceType": "Patient", "id": "259771", "meta": {"versionId": "1", "lastUpdated": "2026-10-23T15:16:44.044-04:00", "source": "#lasm2RvmnfV0drvG"}, "text": {"status": "generated", "div": "<div xmlns=\"http://www.w3.org/1999/xhtml\"><p><b>Generated Narrative with Details</b></p><p><b>Patient</b>: Lerr, Todd</p><p><b>Gender</b>: male</p><p><b>Birthday</b>: 2009-06-07</p><p><b>Phone</b>: 5557259890</p></div>"}, "extension": [{"url": 

In [115]:

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, ArrayType, MapType
)

patient_folder = (
    "Files/raw/Patient/load_date=2026-10-25/"
    "run_id=cf3a17d3-c298-4c0b-80d3-8e17f392d522"
)

bundle_schema = StructType([
    StructField("resourceType", StringType(), True),
    StructField("entry", ArrayType(
        StructType([
            StructField("fullUrl", StringType(), True),
            StructField("resource", MapType(StringType(), StringType()), True)
        ])
    ), True)
])

raw_bundles = (
    spark.read
    .option("multiLine", True)
    .json(f"{patient_folder}/*.json")
)

raw_bundles.select(
    F.count("*").alias("bundle_pages"),
    F.sum(F.size("entry")).alias("bundle_entries")
).show(truncate=False)

display(raw_bundles.select("resourceType", F.size("entry").alias("entries_per_page")))


StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 117, Finished, Available, Finished, False)

+------------+--------------+
|bundle_pages|bundle_entries|
+------------+--------------+
|7           |638           |
+------------+--------------+



SynapseWidget(Synapse.DataFrame, 1887f6e9-8938-42a6-8b4a-ae123de36eeb)

In [116]:

from pyspark.sql import functions as F

patient_entries = (
    raw_bundles
    .select(F.explode("entry").alias("e"))
    .select(
        F.col("e.fullUrl").alias("full_url"),
        F.col("e.resource.id").cast("string").alias("resource_id"),
        F.col("e.resource.resourceType").cast("string").alias("resource_type"),
        F.col("e.resource").alias("resource")
    )
    .filter(F.col("resource_id").isNotNull())
    .dropDuplicates(["resource_id"])
)

bronze_patient_ids = (
    spark.table("Bronze_Patient")
    .select("resource_id")
    .dropDuplicates()
)

new_patient_ids = patient_entries.join(
    bronze_patient_ids,
    on="resource_id",
    how="left_anti"
)

existing_patient_ids = patient_entries.join(
    bronze_patient_ids,
    on="resource_id",
    how="inner"
)

print("Incremental unique Patient IDs:", patient_entries.count())
print("New IDs not in Bronze:", new_patient_ids.count())
print("IDs already in Bronze:", existing_patient_ids.count())

display(new_patient_ids.select("resource_id").limit(10))


StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 118, Finished, Available, Finished, False)

Incremental unique Patient IDs: 638
New IDs not in Bronze: 13
IDs already in Bronze: 625


SynapseWidget(Synapse.DataFrame, efecdba3-68ae-45f0-90ad-cd36bb4c3ada)

In [117]:

print("Bronze_Patient schema:")
spark.table("Bronze_Patient").printSchema()

print("Incremental Patient schema:")
patient_entries.printSchema()


StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 119, Finished, Available, Finished, False)

Bronze_Patient schema:
root
 |-- full_url: string (nullable = true)
 |-- resource: struct (nullable = true)
 |    |-- _birthDate: struct (nullable = true)
 |    |    |-- extension: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- url: string (nullable = true)
 |    |    |    |    |-- valueDateTime: string (nullable = true)
 |    |-- _gender: struct (nullable = true)
 |    |    |-- extension: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- url: string (nullable = true)
 |    |    |    |    |-- valueCode: string (nullable = true)
 |    |    |    |    |-- valueCodeableConcept: struct (nullable = true)
 |    |    |    |    |    |-- coding: array (nullable = true)
 |    |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |    |-- code: string (nullable = true)
 |    |    |    |    |    |    |    |-- display: string (nullable = tru

In [118]:

bronze_patient = spark.table("Bronze_Patient")

print("Bronze resource type:")
print(bronze_patient.schema["resource"].dataType)

print("\nRaw incremental resource type:")
print(raw_bundles.schema["entry"].dataType)


StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 120, Finished, Available, Finished, False)

Bronze resource type:
StructType([StructField('_birthDate', StructType([StructField('extension', ArrayType(StructType([StructField('url', StringType(), True), StructField('valueDateTime', StringType(), True)]), True), True)]), True), StructField('_gender', StructType([StructField('extension', ArrayType(StructType([StructField('url', StringType(), True), StructField('valueCode', StringType(), True), StructField('valueCodeableConcept', StructType([StructField('coding', ArrayType(StructType([StructField('code', StringType(), True), StructField('display', StringType(), True), StructField('system', StringType(), True)]), True), True)]), True)]), True), True)]), True), StructField('active', BooleanType(), True), StructField('address', ArrayType(StructType([StructField('_line', ArrayType(StructType([StructField('extension', ArrayType(StructType([StructField('url', StringType(), True), StructField('valueString', StringType(), True)]), True), True)]), True), True), StructField('city', StringTyp

In [119]:

from pyspark.sql import functions as F

bronze_patient = spark.table("Bronze_Patient")

# Select the raw Patient entries using the inferred nested schema
patient_entries_struct = (
    raw_bundles
    .select(F.explode("entry").alias("e"))
    .select(
        F.col("e.fullUrl").alias("full_url"),
        F.col("e.resource.id").cast("string").alias("resource_id"),
        F.col("e.resource.resourceType").cast("string").alias("resource_type"),
        F.col("e.resource").alias("resource")
    )
    .filter(F.col("resource_id").isNotNull())
    .dropDuplicates(["resource_id"])
)

# Find only Patient IDs that do not already exist in Bronze
bronze_ids = bronze_patient.select("resource_id").distinct()

new_patient_entries = patient_entries_struct.join(
    bronze_ids,
    on="resource_id",
    how="left_anti"
)

# Match the resource data type to the existing Bronze table
new_patient_entries = new_patient_entries.withColumn(
    "resource",
    F.from_json(
        F.to_json(F.col("resource")),
        bronze_patient.schema["resource"].dataType
    )
)

print("New Patient records prepared:", new_patient_entries.count())
print("Records with null resource_id:",
      new_patient_entries.filter(F.col("resource_id").isNull()).count())
print("Records with null resource:",
      new_patient_entries.filter(F.col("resource").isNull()).count())

new_patient_entries.select(
    "full_url", "resource_id", "resource_type"
).show(10, truncate=False)


StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 121, Finished, Available, Finished, False)

New Patient records prepared: 13
Records with null resource_id: 0
Records with null resource: 0
+-------------------------------------------+-----------+-------------+
|full_url                                   |resource_id|resource_type|
+-------------------------------------------+-----------+-------------+
|https://hapi.fhir.org/baseR4/Patient/270314|270314     |Patient      |
|https://hapi.fhir.org/baseR4/Patient/270317|270317     |Patient      |
|https://hapi.fhir.org/baseR4/Patient/270318|270318     |Patient      |
|https://hapi.fhir.org/baseR4/Patient/270319|270319     |Patient      |
|https://hapi.fhir.org/baseR4/Patient/270320|270320     |Patient      |
|https://hapi.fhir.org/baseR4/Patient/270321|270321     |Patient      |
|https://hapi.fhir.org/baseR4/Patient/270323|270323     |Patient      |
|https://hapi.fhir.org/baseR4/Patient/270324|270324     |Patient      |
|https://hapi.fhir.org/baseR4/Patient/270325|270325     |Patient      |
|https://hapi.fhir.org/baseR4/Patient/27

In [120]:
bronze_patient = spark.table("Bronze_Patient")

print("Bronze_Patient schema:")
for field in bronze_patient.schema.fields:
    print(f"{field.name}: {field.dataType.simpleString()}")

print("\nExisting sample record:")
bronze_patient.select(*bronze_patient.columns).limit(1).show(
    vertical=True,
    truncate=False
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 122, Finished, Available, Finished, False)

Bronze_Patient schema:
full_url: string
resource: struct<_birthDate:struct<extension:array<struct<url:string,valueDateTime:string>>>,_gender:struct<extension:array<struct<url:string,valueCode:string,valueCodeableConcept:struct<coding:array<struct<code:string,display:string,system:string>>>>>>,active:boolean,address:array<struct<_line:array<struct<extension:array<struct<url:string,valueString:string>>>>,city:string,country:string,district:string,extension:array<struct<extension:array<struct<url:string,valueDecimal:double,valueString:string>>,url:string,valueCode:string,valueCodeableConcept:struct<coding:array<struct<code:string,system:string>>>,valueCoding:struct<code:string,system:string>,valueString:string>>,id:string,line:array<string>,period:struct<end:string,start:string>,postalCode:string,state:string,text:string,type:string,use:string>>,birthDate:string,communication:array<struct<language:struct<coding:array<struct<code:string,display:string,system:string,userSelected:boolean>>,t

In [121]:
from pyspark.sql import functions as F
from datetime import datetime, timezone, timedelta

bronze_patient = spark.table("Bronze_Patient")

# Match the nested resource schema to Bronze_Patient
prepared_patient = new_patient_entries.withColumn(
    "resource",
    F.from_json(
        F.to_json(F.col("resource")),
        bronze_patient.schema["resource"].dataType
    )
)

# Add ingestion metadata
cutoff = (
    datetime.now(timezone.utc) - timedelta(days=2)
).strftime("%Y-%m-%dT%H:%M:%SZ")

prepared_patient = (
    prepared_patient
    .withColumn("extraction_timestamp", F.current_timestamp())
    .withColumn("api_url_or_params", F.lit(
        f"https://hapi.fhir.org/baseR4/Patient?_lastUpdated=ge{cutoff}"
    ))
)

# Match Bronze column order exactly
prepared_patient = prepared_patient.select(
    *bronze_patient.columns
)

print("Prepared rows:", prepared_patient.count())
print("Prepared schema matches Bronze:",
      prepared_patient.schema == bronze_patient.schema)

print("Duplicate IDs in prepared data:",
      prepared_patient.groupBy("resource_id")
      .count()
      .filter(F.col("count") > 1)
      .count())

prepared_patient.select(
    "full_url",
    "resource_id",
    "resource_type",
    "extraction_timestamp",
    "api_url_or_params"
).show(5, truncate=False)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 123, Finished, Available, Finished, False)

Prepared rows: 13
Prepared schema matches Bronze: False
Duplicate IDs in prepared data: 0
+-------------------------------------------+-----------+-------------+--------------------------+------------------------------------------------------------------------+
|full_url                                   |resource_id|resource_type|extraction_timestamp      |api_url_or_params                                                       |
+-------------------------------------------+-----------+-------------+--------------------------+------------------------------------------------------------------------+
|https://hapi.fhir.org/baseR4/Patient/270314|270314     |Patient      |2026-10-25 19:40:22.641479|https://hapi.fhir.org/baseR4/Patient?_lastUpdated=ge2026-10-23T19:40:21Z|
|https://hapi.fhir.org/baseR4/Patient/270317|270317     |Patient      |2026-10-25 19:40:22.641479|https://hapi.fhir.org/baseR4/Patient?_lastUpdated=ge2026-10-23T19:40:21Z|
|https://hapi.fhir.org/baseR4/Patient/270318|27031

In [122]:
bronze_schema = spark.table("Bronze_Patient").schema
prepared_schema = prepared_patient.schema

print("Column names match:",
      bronze_schema.fieldNames() == prepared_schema.fieldNames())

print("\nColumn-by-column type comparison:")
for bronze_field, prepared_field in zip(
    bronze_schema.fields, prepared_schema.fields
):
    print(
        f"{bronze_field.name}: "
        f"Bronze={bronze_field.dataType.simpleString() == prepared_field.dataType.simpleString()}"
    )

print("\nDetailed resource schema differences:")
bronze_resource_schema = bronze_schema["resource"].dataType
prepared_resource_schema = prepared_schema["resource"].dataType

for bronze_field, prepared_field in zip(
    bronze_resource_schema.fields, prepared_resource_schema.fields
):
    if bronze_field.dataType != prepared_field.dataType:
        print(
            f"{bronze_field.name}: "
            f"Bronze={bronze_field.dataType.simpleString()} | "
            f"Prepared={prepared_field.dataType.simpleString()}"
        )

print("\nNumber of top-level columns:",
      len(bronze_schema.fields), len(prepared_schema.fields))
print("Number of resource fields:",
      len(bronze_resource_schema.fields),
      len(prepared_resource_schema.fields))

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 124, Finished, Available, Finished, False)

Column names match: True

Column-by-column type comparison:
full_url: Bronze=True
resource: Bronze=True
extraction_timestamp: Bronze=True
resource_id: Bronze=True
resource_type: Bronze=True
api_url_or_params: Bronze=True

Detailed resource schema differences:

Number of top-level columns: 6 6
Number of resource fields: 25 25


In [123]:
bronze_schema = spark.table("Bronze_Patient").schema
prepared_schema = prepared_patient.schema

print("Exact schema equality:", bronze_schema == prepared_schema)
print("Same data types:", bronze_schema.jsonValue()["fields"] == prepared_schema.jsonValue()["fields"])

print("\nTop-level field details:")
for b, p in zip(bronze_schema.fields, prepared_schema.fields):
    print(
        f"{b.name}: "
        f"Bronze nullable={b.nullable}, Prepared nullable={p.nullable}, "
        f"metadata_equal={b.metadata == p.metadata}"
    )

print("\nPrepared record validation:")
print("Rows:", prepared_patient.count())
print("Null IDs:", prepared_patient.filter(F.col("resource_id").isNull()).count())
print("Null resources:", prepared_patient.filter(F.col("resource").isNull()).count())
print(
    "IDs already in Bronze:",
    prepared_patient.select("resource_id").join(
        spark.table("Bronze_Patient").select("resource_id").distinct(),
        "resource_id",
        "inner"
    ).count()
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 125, Finished, Available, Finished, False)

Exact schema equality: False
Same data types: False

Top-level field details:
full_url: Bronze nullable=True, Prepared nullable=True, metadata_equal=True
resource: Bronze nullable=True, Prepared nullable=True, metadata_equal=True
extraction_timestamp: Bronze nullable=True, Prepared nullable=False, metadata_equal=True
resource_id: Bronze nullable=True, Prepared nullable=True, metadata_equal=True
resource_type: Bronze nullable=True, Prepared nullable=True, metadata_equal=True
api_url_or_params: Bronze nullable=True, Prepared nullable=False, metadata_equal=True

Prepared record validation:
Rows: 13
Null IDs: 0
Null resources: 0
IDs already in Bronze: 0


In [124]:
from pyspark.sql import functions as F

bronze_patient = spark.table("Bronze_Patient")
bronze_schema = bronze_patient.schema

# Build a DataFrame using Bronze's exact schema
prepared_patient_aligned = spark.createDataFrame(
    prepared_patient.select(*bronze_patient.columns).rdd,
    schema=bronze_schema
)

print("Exact schema equality:",
      prepared_patient_aligned.schema == bronze_schema)

print("Prepared rows:", prepared_patient_aligned.count())

print("Null IDs:",
      prepared_patient_aligned.filter(
          F.col("resource_id").isNull()
      ).count())

print("Duplicate IDs:",
      prepared_patient_aligned.groupBy("resource_id")
      .count()
      .filter(F.col("count") > 1)
      .count())

print("IDs already in Bronze:",
      prepared_patient_aligned.select("resource_id")
      .join(
          bronze_patient.select("resource_id").distinct(),
          "resource_id",
          "inner"
      ).count())

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 126, Finished, Available, Finished, False)

Exact schema equality: True
Prepared rows: 13
Null IDs: 0
Duplicate IDs: 0
IDs already in Bronze: 0


In [125]:
# Append only the 13 validated, previously unseen Patient records
prepared_patient_aligned.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("Bronze_Patient")

print("Append completed.")

print(
    "Bronze_Patient row count:",
    spark.table("Bronze_Patient").count()
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 127, Finished, Available, Finished, False)

Append completed.
Bronze_Patient row count: 9054


In [126]:
from pyspark.sql import functions as F

bronze_patient = spark.table("Bronze_Patient")

inserted_ids = prepared_patient_aligned.select("resource_id").distinct()

verified_records = bronze_patient.join(
    inserted_ids,
    on="resource_id",
    how="inner"
)

print("Expected new records:", inserted_ids.count())
print("Records found in Bronze:", verified_records.count())

print(
    "Duplicate IDs among inserted records:",
    verified_records.groupBy("resource_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Total Bronze_Patient rows:", bronze_patient.count())

verified_records.select(
    "resource_id",
    "resource_type",
    "extraction_timestamp",
    "api_url_or_params"
).show(15, truncate=False)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 128, Finished, Available, Finished, False)

Expected new records: 13
Records found in Bronze: 13
Duplicate IDs among inserted records: 0
Total Bronze_Patient rows: 9054
+------------+-------------+--------------------------+------------------------------------------------------------------------+
|resource_id |resource_type|extraction_timestamp      |api_url_or_params                                                       |
+------------+-------------+--------------------------+------------------------------------------------------------------------+
|270336      |Patient      |2026-10-25 19:42:26.987896|https://hapi.fhir.org/baseR4/Patient?_lastUpdated=ge2026-10-23T19:40:21Z|
|270324      |Patient      |2026-10-25 19:42:26.987896|https://hapi.fhir.org/baseR4/Patient?_lastUpdated=ge2026-10-23T19:40:21Z|
|270318      |Patient      |2026-10-25 19:42:26.987896|https://hapi.fhir.org/baseR4/Patient?_lastUpdated=ge2026-10-23T19:40:21Z|
|270372      |Patient      |2026-10-25 19:42:26.987896|https://hapi.fhir.org/baseR4/Patient?_lastUpda

In [127]:
silver_patient = spark.table("Silver_Patient")

print("Silver_Patient schema:")
silver_patient.printSchema()

print("Current Silver rows:", silver_patient.count())
print(
    "Distinct Silver Patient IDs:",
    silver_patient.select("resource_id").distinct().count()
)

print("New IDs missing from Silver:")
new_missing_silver = (
    prepared_patient_aligned.select("resource_id").distinct()
    .join(
        silver_patient.select("resource_id").distinct(),
        on="resource_id",
        how="left_anti"
    )
)

print(new_missing_silver.count())
new_missing_silver.show(20, truncate=False)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 129, Finished, Available, Finished, False)

Silver_Patient schema:
root
 |-- full_url: string (nullable = true)
 |-- resource: struct (nullable = true)
 |    |-- _birthDate: struct (nullable = true)
 |    |    |-- extension: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- url: string (nullable = true)
 |    |    |    |    |-- valueDateTime: string (nullable = true)
 |    |-- _gender: struct (nullable = true)
 |    |    |-- extension: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- url: string (nullable = true)
 |    |    |    |    |-- valueCode: string (nullable = true)
 |    |    |    |    |-- valueCodeableConcept: struct (nullable = true)
 |    |    |    |    |    |-- coding: array (nullable = true)
 |    |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |    |-- code: string (nullable = true)
 |    |    |    |    |    |    |    |-- display: string (nullable = tru

In [128]:
from pyspark.sql import functions as F

silver_patient = spark.table("Silver_Patient")

# Keep only records whose IDs are not already in Silver
silver_existing_ids = silver_patient.select("resource_id").distinct()

new_silver_records = (
    prepared_patient_aligned
    .join(silver_existing_ids, on="resource_id", how="left_anti")
    .withColumn("silver_processed_timestamp", F.current_timestamp())
)

# Match the existing Silver schema and column order
new_silver_records = spark.createDataFrame(
    new_silver_records.select(*silver_patient.columns).rdd,
    schema=silver_patient.schema
)

print("New records prepared:", new_silver_records.count())
print(
    "Schema matches Silver:",
    new_silver_records.schema == silver_patient.schema
)
print(
    "Null resource IDs:",
    new_silver_records.filter(F.col("resource_id").isNull()).count()
)
print(
    "Duplicate IDs in prepared records:",
    new_silver_records.groupBy("resource_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
print(
    "IDs already in Silver:",
    new_silver_records.select("resource_id")
    .join(
        silver_patient.select("resource_id").distinct(),
        on="resource_id",
        how="inner"
    )
    .count()
)

new_silver_records.select(
    "resource_id",
    "resource_type",
    "extraction_timestamp",
    "silver_processed_timestamp"
).show(15, truncate=False)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 130, Finished, Available, Finished, False)

New records prepared: 13
Schema matches Silver: True
Null resource IDs: 0
Duplicate IDs in prepared records: 0
IDs already in Silver: 0
+------------+-------------+--------------------------+--------------------------+
|resource_id |resource_type|extraction_timestamp      |silver_processed_timestamp|
+------------+-------------+--------------------------+--------------------------+
|270314      |Patient      |2026-10-25 19:42:26.987896|2026-10-25 19:46:06.808449|
|270317      |Patient      |2026-10-25 19:42:26.987896|2026-10-25 19:46:06.808449|
|270318      |Patient      |2026-10-25 19:42:26.987896|2026-10-25 19:46:06.808449|
|270319      |Patient      |2026-10-25 19:42:26.987896|2026-10-25 19:46:06.808449|
|270320      |Patient      |2026-10-25 19:42:26.987896|2026-10-25 19:46:06.808449|
|270321      |Patient      |2026-10-25 19:42:26.987896|2026-10-25 19:46:06.808449|
|270323      |Patient      |2026-10-25 19:42:26.987896|2026-10-25 19:46:06.808449|
|270324      |Patient      |2026-1

In [129]:
new_silver_records.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("Silver_Patient")

silver_after = spark.table("Silver_Patient")

print("Append completed.")
print("Silver_Patient total rows:", silver_after.count())
print(
    "Distinct Patient IDs:",
    silver_after.select("resource_id").distinct().count()
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 131, Finished, Available, Finished, False)

Append completed.
Silver_Patient total rows: 9044
Distinct Patient IDs: 9044


In [130]:
gold_summary = spark.table("Gold_Healthcare_Summary")

print("Gold_Healthcare_Summary schema:")
gold_summary.printSchema()

print("Current summary data:")
display(gold_summary)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 132, Finished, Available, Finished, False)

Gold_Healthcare_Summary schema:
root
 |-- resource_name: string (nullable = true)
 |-- total_records: long (nullable = true)
 |-- report_generated_timestamp: timestamp (nullable = true)

Current summary data:


SynapseWidget(Synapse.DataFrame, cb07f002-f714-4be4-bb4b-e8fd86f36410)

In [131]:
from pyspark.sql import functions as F

resource_tables = [
    ("Patient", "Silver_Patient"),
    ("Encounter", "Silver_Encounter"),
    ("Observation", "Silver_Observation"),
    ("Condition", "Silver_Condition")
]

summary_rows = []

for resource_name, table_name in resource_tables:
    total_records = spark.table(table_name).count()
    summary_rows.append((resource_name, total_records))

summary_schema = """
    resource_name STRING,
    total_records LONG
"""

refreshed_summary = spark.createDataFrame(
    summary_rows,
    schema=summary_schema
).withColumn(
    "report_generated_timestamp",
    F.current_timestamp()
)

display(refreshed_summary.orderBy("resource_name"))

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 133, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 658cf4ad-9d4c-49d4-87a5-b41bd5f0d009)

In [132]:
refreshed_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("Gold_Healthcare_Summary")

print("Gold summary refreshed successfully.")

display(
    spark.table("Gold_Healthcare_Summary")
    .orderBy("resource_name")
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 134, Finished, Available, Finished, False)

Gold summary refreshed successfully.


SynapseWidget(Synapse.DataFrame, f7cb0ac9-d2e2-4aab-a332-386c2cd380c5)

In [133]:
from pyspark.sql import functions as F

history_df = spark.table("Gold_Patient_History")

print("Gold_Patient_History schema:")
history_df.printSchema()

print("Total history versions:", history_df.count())

print(
    "Current versions:",
    history_df.filter(F.col("valid_to").isNull()).count()
)

print(
    "Historical versions:",
    history_df.filter(F.col("valid_to").isNotNull()).count()
)

print(
    "Duplicate current Patient IDs:",
    history_df
        .filter(F.col("valid_to").isNull())
        .groupBy("resource_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
)

display(history_df.limit(5))

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 136, Finished, Available, Finished, False)

Gold_Patient_History schema:
root
 |-- full_url: string (nullable = true)
 |-- resource: struct (nullable = true)
 |    |-- _birthDate: struct (nullable = true)
 |    |    |-- extension: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- url: string (nullable = true)
 |    |    |    |    |-- valueDateTime: string (nullable = true)
 |    |-- _gender: struct (nullable = true)
 |    |    |-- extension: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- url: string (nullable = true)
 |    |    |    |    |-- valueCode: string (nullable = true)
 |    |    |    |    |-- valueCodeableConcept: struct (nullable = true)
 |    |    |    |    |    |-- coding: array (nullable = true)
 |    |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |    |-- code: string (nullable = true)
 |    |    |    |    |    |    |    |-- display: string (nullable

SynapseWidget(Synapse.DataFrame, ffd41043-6efa-4e4c-9aa1-589d959ef3b7)

In [134]:
from pyspark.sql import functions as F

history_df = spark.table("Gold_Patient_History")

print("History columns:")
print(history_df.columns)

print("Total history rows:", history_df.count())

print(
    "Current versions:",
    history_df.filter(F.col("valid_to").isNull()).count()
)

# Identify the 13 newly ingested Patient IDs
new_patient_ids = [
    row["resource_id"]
    for row in new_patient_entries.select("resource_id")
        .distinct()
        .collect()
]

# Check which new IDs already exist in history
existing_history_ids = (
    history_df
    .filter(F.col("resource_id").isin(new_patient_ids))
    .select("resource_id")
    .distinct()
)

print("New Patient IDs prepared:", len(new_patient_ids))
print("Already present in history:", existing_history_ids.count())
print(
    "Not yet present in history:",
    len(new_patient_ids) - existing_history_ids.count()
)

display(existing_history_ids)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 137, Finished, Available, Finished, False)

History columns:
['full_url', 'resource', 'extraction_timestamp', 'resource_id', 'resource_type', 'api_url_or_params', 'silver_processed_timestamp', 'is_current', 'valid_from', 'valid_to', 'record_hash']
Total history rows: 9031
Current versions: 9031
New Patient IDs prepared: 0
Already present in history: 0
Not yet present in history: 0


SynapseWidget(Synapse.DataFrame, fa770985-5d0f-4070-a4b1-36911a293110)

In [135]:
from pyspark.sql import functions as F

silver_patient = spark.table("Silver_Patient")
history_df = spark.table("Gold_Patient_History")

history_ids = history_df.select("resource_id").distinct()

patients_missing_from_history = (
    silver_patient
    .join(history_ids, on="resource_id", how="left_anti")
)

print(
    "Patients in Silver:",
    silver_patient.select("resource_id").distinct().count()
)

print(
    "Patients missing from history:",
    patients_missing_from_history.select("resource_id").distinct().count()
)

print("Missing Patient IDs:")
display(
    patients_missing_from_history.select(
        "resource_id", "resource_type"
    ).orderBy("resource_id")
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 139, Finished, Available, Finished, False)

Patients in Silver: 9044
Patients missing from history: 13
Missing Patient IDs:


SynapseWidget(Synapse.DataFrame, da9816d3-1bf5-4be7-8b99-8705a5c58e57)

In [136]:
from pyspark.sql import functions as F

history_df = spark.table("Gold_Patient_History")

history_ids = history_df.select("resource_id").distinct()

# Find Silver Patients that do not yet exist in history
missing_patients = (
    spark.table("Silver_Patient")
    .join(history_ids, on="resource_id", how="left_anti")
)

# Prepare each new Patient as its first current history version
new_history_records = (
    missing_patients
    .withColumn("is_current", F.lit(True))
    .withColumn("valid_from", F.current_timestamp())
    .withColumn("valid_to", F.lit(None).cast("timestamp"))
    .withColumn(
        "record_hash",
        F.sha2(F.to_json(F.col("resource")), 256)
    )
)

# Align the exact production history schema
new_history_records = spark.createDataFrame(
    new_history_records.select(*history_df.columns).rdd,
    schema=history_df.schema
)

print("Prepared new history records:", new_history_records.count())
print(
    "Schema matches history:",
    new_history_records.schema == history_df.schema
)
print(
    "Null Patient IDs:",
    new_history_records.filter(F.col("resource_id").isNull()).count()
)
print(
    "Duplicate Patient IDs:",
    new_history_records.groupBy("resource_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

display(
    new_history_records.select(
        "resource_id", "is_current", "valid_from",
        "valid_to", "record_hash"
    ).orderBy("resource_id")
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 140, Finished, Available, Finished, False)

Prepared new history records: 13
Schema matches history: True
Null Patient IDs: 0
Duplicate Patient IDs: 0


SynapseWidget(Synapse.DataFrame, fe98ffe5-342a-4712-a24d-2619af526028)

In [137]:
from pyspark.sql import functions as F

history_df = spark.table("Gold_Patient_History")

prepared_count = new_history_records.count()

already_in_history = (
    new_history_records.select("resource_id").distinct()
    .join(
        history_df.select("resource_id").distinct(),
        on="resource_id",
        how="inner"
    )
    .count()
)

invalid_records = new_history_records.filter(
    F.col("resource_id").isNull()
    | F.col("is_current").isNull()
    | F.col("valid_from").isNull()
    | F.col("valid_to").isNotNull()
    | F.col("record_hash").isNull()
).count()

print("Prepared records:", prepared_count)
print("Schema matches:", new_history_records.schema == history_df.schema)
print("IDs already in history:", already_in_history)
print("Invalid prepared records:", invalid_records)
print("Existing history rows:", history_df.count())

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 141, Finished, Available, Finished, False)

Prepared records: 13
Schema matches: True
IDs already in history: 0
Invalid prepared records: 0
Existing history rows: 9031


In [138]:
# Append the 13 validated first-time Patient history records
new_history_records.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("Gold_Patient_History")

# Verify the result
history_after = spark.table("Gold_Patient_History")

print("Append completed.")
print("Total history rows:", history_after.count())

print(
    "Current versions:",
    history_after.filter(F.col("is_current") == True).count()
)

print(
    "Historical versions:",
    history_after.filter(F.col("is_current") == False).count()
)

print(
    "Duplicate current Patient IDs:",
    history_after
        .filter(F.col("is_current") == True)
        .groupBy("resource_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 143, Finished, Available, Finished, False)

Append completed.
Total history rows: 9044
Current versions: 9044
Historical versions: 0
Duplicate current Patient IDs: 0


In [139]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Incremental Patient API run folder
patient_folder = (
    "Files/raw/Patient/load_date=2026-10-25/"
    "run_id=cf3a17d3-c298-4c0b-80d3-8e17f392d522"
)

# Read incremental raw bundles
raw_bundles = (
    spark.read
    .option("multiLine", True)
    .json(f"{patient_folder}/*.json")
)

incremental_entries = (
    raw_bundles
    .select(F.explode_outer("entry").alias("entry"))
    .filter(F.col("entry.resource.resourceType") == "Patient")
    .select(
        F.col("entry.resource.id").alias("resource_id"),
        F.col("entry.resource").alias("resource")
    )
    .filter(F.col("resource_id").isNotNull())
)

# Align resource structure with Bronze schema before hashing
bronze_patient = spark.table("Bronze_Patient")
resource_schema = bronze_patient.schema["resource"].dataType

incremental_entries = incremental_entries.withColumn(
    "resource",
    F.from_json(F.to_json(F.col("resource")), resource_schema)
)

incremental_entries = incremental_entries.withColumn(
    "record_hash",
    F.sha2(F.to_json(F.col("resource")), 256)
)

# Deduplicate incremental records by ID
# If repeated, keep the record with the latest FHIR meta.lastUpdated
window_spec = Window.partitionBy("resource_id").orderBy(
    F.col("resource.meta.lastUpdated").desc_nulls_last()
)

incremental_latest = (
    incremental_entries
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# Compute hashes for existing Bronze records
bronze_hashed = (
    bronze_patient
    .filter(F.col("resource_id").isNotNull())
    .withColumn(
        "record_hash",
        F.sha2(F.to_json(F.col("resource")), 256)
    )
    .select("resource_id", F.col("record_hash").alias("bronze_hash"))
    .dropDuplicates(["resource_id"])
)

# Compare incremental data against Bronze
comparison = (
    incremental_latest.alias("i")
    .join(bronze_hashed.alias("b"), on="resource_id", how="left")
    .withColumn(
        "comparison_status",
        F.when(F.col("b.bronze_hash").isNull(), "NEW_TO_BRONZE")
         .when(F.col("i.record_hash") != F.col("b.bronze_hash"), "CHANGED")
         .otherwise("UNCHANGED")
    )
)

print("Unique incremental Patient IDs:", incremental_latest.count())
print("New to Bronze:", comparison.filter(F.col("comparison_status") == "NEW_TO_BRONZE").count())
print("Changed existing Patients:", comparison.filter(F.col("comparison_status") == "CHANGED").count())
print("Unchanged existing Patients:", comparison.filter(F.col("comparison_status") == "UNCHANGED").count())

display(
    comparison
    .groupBy("comparison_status")
    .count()
    .orderBy("comparison_status")
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 144, Finished, Available, Finished, False)

Unique incremental Patient IDs: 638
New to Bronze: 0
Changed existing Patients: 0
Unchanged existing Patients: 638


SynapseWidget(Synapse.DataFrame, fbedbe82-ec28-4a7d-8d47-7eb17a81dc8c)

In [140]:
# Current Patient versions in SCD2 history
history_current = (
    spark.table("Gold_Patient_History")
    .filter(F.col("is_current") == True)
    .select(
        "resource_id",
        F.col("record_hash").alias("history_hash")
    )
    .dropDuplicates(["resource_id"])
)

history_comparison = (
    incremental_latest.alias("i")
    .join(history_current.alias("h"), on="resource_id", how="left")
    .withColumn(
        "comparison_status",
        F.when(F.col("h.history_hash").isNull(), "NEW_TO_HISTORY")
         .when(F.col("i.record_hash") != F.col("h.history_hash"), "CHANGED")
         .otherwise("UNCHANGED")
    )
)

print("Unique incremental Patient IDs:", incremental_latest.count())
print(
    "New to history:",
    history_comparison.filter(
        F.col("comparison_status") == "NEW_TO_HISTORY"
    ).count()
)
print(
    "Changed existing Patients:",
    history_comparison.filter(
        F.col("comparison_status") == "CHANGED"
    ).count()
)
print(
    "Unchanged existing Patients:",
    history_comparison.filter(
        F.col("comparison_status") == "UNCHANGED"
    ).count()
)

display(
    history_comparison
    .groupBy("comparison_status")
    .count()
    .orderBy("comparison_status")
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 145, Finished, Available, Finished, False)

Unique incremental Patient IDs: 638
New to history: 0
Changed existing Patients: 0
Unchanged existing Patients: 638


SynapseWidget(Synapse.DataFrame, 223df68b-8463-4c83-9397-d4f0151dad8e)

In [141]:
from pyspark.sql import functions as F

history_df = spark.table("Gold_Patient_History")

current_df = history_df.filter(F.col("is_current") == True)
historical_df = history_df.filter(F.col("is_current") == False)

print("Total history rows:", history_df.count())
print("Current versions:", current_df.count())
print("Historical versions:", historical_df.count())

print(
    "Duplicate current Patient IDs:",
    current_df.groupBy("resource_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
)

print(
    "Current rows with valid_to populated:",
    current_df.filter(F.col("valid_to").isNotNull()).count()
)

print(
    "Historical rows with valid_to missing:",
    historical_df.filter(F.col("valid_to").isNull()).count()
)

print(
    "Rows missing record_hash:",
    history_df.filter(F.col("record_hash").isNull()).count()
)

print(
    "Rows with invalid validity dates:",
    history_df.filter(
        F.col("valid_from").isNull()
        | (
            F.col("valid_to").isNotNull()
            & (F.col("valid_to") < F.col("valid_from"))
        )
    ).count()
)


StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 147, Finished, Available, Finished, False)

Total history rows: 9044
Current versions: 9044
Historical versions: 0
Duplicate current Patient IDs: 0
Current rows with valid_to populated: 0
Historical rows with valid_to missing: 0
Rows missing record_hash: 0
Rows with invalid validity dates: 0


In [142]:
test_history = spark.table("Test_Patient_History_SCD2")

test_patient_id = "045ed820-8c47-477d-a6d9-25b69e411b7d"

print("Test table total rows:", test_history.count())

display(
    test_history
    .filter(F.col("resource_id") == test_patient_id)
    .select(
        "resource_id",
        F.col("resource.gender").alias("gender"),
        "is_current",
        "valid_from",
        "valid_to",
        "record_hash"
    )
    .orderBy("valid_from")
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 148, Finished, Available, Finished, False)

Test table total rows: 9032


SynapseWidget(Synapse.DataFrame, 2229c5f1-c888-48c4-b806-032596e0842f)

In [143]:
test_history = spark.table("Test_Patient_History_SCD2")

test_patient_id = "045ed820-8c47-477d-a6d9-25b69e411b7d"

display(
    test_history
    .filter(F.col("resource_id") == test_patient_id)
    .select(
        "resource_id",
        F.col("resource.gender").alias("gender"),
        "is_current",
        "valid_from",
        "valid_to",
        "record_hash"
    )
    .orderBy("valid_from")
)

patient_versions = test_history.filter(
    F.col("resource_id") == test_patient_id
)

print("Versions for test Patient:", patient_versions.count())
print(
    "Current versions:",
    patient_versions.filter(F.col("is_current") == True).count()
)
print(
    "Historical versions:",
    patient_versions.filter(F.col("is_current") == False).count()
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 149, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5d8fc146-1293-440b-93d5-36216b69acca)

Versions for test Patient: 2
Current versions: 1
Historical versions: 1


In [144]:
from pyspark.sql import functions as F

test_history = spark.table("Test_Patient_History_SCD2")

duplicate_current_ids = (
    test_history
    .filter(F.col("is_current") == True)
    .groupBy("resource_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Total history rows:", test_history.count())
print(
    "Current versions:",
    test_history.filter(F.col("is_current") == True).count()
)
print(
    "Historical versions:",
    test_history.filter(F.col("is_current") == False).count()
)
print("Duplicate current Patient IDs:", duplicate_current_ids.count())

print(
    "Current rows with valid_to populated:",
    test_history.filter(
        (F.col("is_current") == True) & F.col("valid_to").isNotNull()
    ).count()
)

print(
    "Historical rows with valid_to missing:",
    test_history.filter(
        (F.col("is_current") == False) & F.col("valid_to").isNull()
    ).count()
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 151, Finished, Available, Finished, False)

Total history rows: 9032
Current versions: 9031
Historical versions: 1
Duplicate current Patient IDs: 0
Current rows with valid_to populated: 0
Historical rows with valid_to missing: 0


In [145]:
from pyspark.sql import functions as F

prod_history = spark.table("Gold_Patient_History")

print("Production history rows:", prod_history.count())

print(
    "Current versions:",
    prod_history.filter(F.col("is_current") == True).count()
)

print(
    "Historical versions:",
    prod_history.filter(F.col("is_current") == False).count()
)

print(
    "Duplicate current Patient IDs:",
    prod_history
    .filter(F.col("is_current") == True)
    .groupBy("resource_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 152, Finished, Available, Finished, False)

Production history rows: 9044
Current versions: 9044
Historical versions: 0
Duplicate current Patient IDs: 0


In [146]:
from delta.tables import DeltaTable

history_table = DeltaTable.forName(spark, "Gold_Patient_History")

display(
    history_table.history()
    .select("version", "timestamp", "operation", "operationParameters")
    .orderBy("version", ascending=False)
    .limit(15)
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 153, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6870d6ad-20ba-4167-8760-608b1ad9d64a)

In [147]:
from pyspark.sql import functions as F

# Read the original production history before the 13-row append
history_before = (
    spark.read.format("delta")
    .option("versionAsOf", 0)
    .table("Gold_Patient_History")
)

silver_patients = spark.table("Silver_Patient")

# Calculate the content hash for current Silver records
silver_hashed = silver_patients.withColumn(
    "new_record_hash",
    F.sha2(F.to_json(F.col("resource")), 256)
)

# Compare Silver data with the original history snapshot
history_baseline = history_before.select(
    "resource_id",
    F.col("record_hash").alias("old_record_hash")
)

comparison = silver_hashed.join(
    history_baseline,
    on="resource_id",
    how="left"
)

new_patients = comparison.filter(
    F.col("old_record_hash").isNull()
)

changed_patients = comparison.filter(
    F.col("old_record_hash").isNotNull()
    & (F.col("new_record_hash") != F.col("old_record_hash"))
)

unchanged_patients = comparison.filter(
    F.col("old_record_hash").isNotNull()
    & (F.col("new_record_hash") == F.col("old_record_hash"))
)

print("Original history rows:", history_before.count())
print("Current Silver Patient rows:", silver_patients.count())
print("New Patients since baseline:", new_patients.count())
print("Changed existing Patients:", changed_patients.count())
print("Unchanged existing Patients:", unchanged_patients.count())

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 154, Finished, Available, Finished, False)

Original history rows: 9031
Current Silver Patient rows: 9044
New Patients since baseline: 13
Changed existing Patients: 0
Unchanged existing Patients: 9031


In [148]:
from pyspark.sql import functions as F

prod_history = spark.table("Gold_Patient_History")

silver_hashes = (
    spark.table("Silver_Patient")
    .withColumn(
        "silver_hash",
        F.sha2(F.to_json(F.col("resource")), 256)
    )
    .select("resource_id", "silver_hash")
)

current_history = (
    prod_history
    .filter(F.col("is_current") == True)
    .select(
        "resource_id",
        F.col("record_hash").alias("history_hash")
    )
)

validation = silver_hashes.join(
    current_history,
    on="resource_id",
    how="full_outer"
)

print(
    "Silver Patients missing from current history:",
    validation.filter(F.col("silver_hash").isNotNull() & F.col("history_hash").isNull()).count()
)

print(
    "Current history Patients missing from Silver:",
    validation.filter(F.col("silver_hash").isNull() & F.col("history_hash").isNotNull()).count()
)

print(
    "Hash mismatches:",
    validation.filter(
        F.col("silver_hash").isNotNull()
        & F.col("history_hash").isNotNull()
        & (F.col("silver_hash") != F.col("history_hash"))
    ).count()
)

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 155, Finished, Available, Finished, False)

Silver Patients missing from current history: 0
Current history Patients missing from Silver: 0
Hash mismatches: 0


In [151]:
from pyspark.sql import functions as F

silver_tables = [
    ("Patient", "Silver_Patient"),
    ("Encounter", "Silver_Encounter"),
    ("Observation", "Silver_Observation"),
    ("Condition", "Silver_Condition")
]

gold_summary = spark.table("Gold_Healthcare_Summary")

for resource_name, table_name in silver_tables:
    silver_count = spark.table(table_name).count()

    gold_row = (
        gold_summary
        .filter(F.col("resource_name") == resource_name)
        .select("total_records")
        .first()
    )

    gold_count = gold_row["total_records"] if gold_row else None

    print(
        f"{resource_name}: Silver={silver_count}, "
        f"Gold={gold_count}, Match={silver_count == gold_count}"
    )

StatementMeta(, bacc22e6-2ddd-4df2-afa1-1d76c82fc6ce, 160, Finished, Available, Finished, False)

Patient: Silver=9044, Gold=9044, Match=True
Encounter: Silver=10603, Gold=10603, Match=True
Observation: Silver=58122, Gold=58122, Match=True
Condition: Silver=5944, Gold=5944, Match=True
